In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 0 — Setup & Imports
# ══════════════════════════════════════════════════════════
import importlib, sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# --- Drive Mount ---
# from google.colab import drive
# drive.mount('/content/drive')
src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

for mod_name in ['plot_style', 'cds_config', 'm1_calibration', 'm2_thresholds', 'm3_errors']:
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(src_dir, f"{mod_name}.py"))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)

from cds_config import *
setup_notebook("Chat 4 — M4 E2E Pipeline Analysis")
from m1_calibration import load_calibrated_probs
from m2_thresholds import load_threshold_config, apply_stage1_threshold, assign_confidence_zone, get_operating_point
from m3_errors import get_misclassified, confusion_summary, clinical_risk_score

# Create M4 directories
for sub in ['analysis', 'plots', 'reports']:
    os.makedirs(PATHS.module_dir('m4_e2e', sub), exist_ok=True)

# Load threshold config
th_config = load_threshold_config(PATHS)

# Color palette & colormap
cmap_tufte = LinearSegmentedColormap.from_list(
    'tufte_cm', ['#FFFFFF', PALETTE['base1']], N=256)

print("✅ M4 Setup complete")
print(f"   Module dir: {PATHS.module_dir('m4_e2e')}")
for sc in SCENARIOS:
    op = get_operating_point(th_config, sc)
    print(f"   {sc}: S1 th={op['s1_threshold']}, S2 LOW<{op['s2_zone_low']}, S2 HIGH≥{op['s2_zone_high']}")

## CELL 1 — E2E Pipeline Simulation (OOF + Test)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 1 — E2E Pipeline Simulation (Final)
# ══════════════════════════════════════════════════════════

def simulate_e2e_pipeline(paths, scenario, split, th_config):
    """
    E2E pipeline simulation — S1-S2 merge via positional alignment.
    S1 filter → routing to S2 → 5-class E2E label.
    """
    op = get_operating_point(th_config, scenario)
    s1_th  = op['s1_threshold']
    s2_low = op['s2_zone_low']
    s2_high = op['s2_zone_high']

    # ── Load S1 ──
    s1_raw = load_calibrated_probs(paths, '1', scenario, split)
    prob_col_s1 = 'prob_IAS_cal' if 'prob_IAS_cal' in s1_raw.columns else 'prob_IAS'

    e2e = pd.DataFrame({
        'idx_s1':     s1_raw.index,
        's1_true':    s1_raw['target'].map({1: 'IAS', 0: 'DAS'}),
        'diagnosis':  s1_raw['diagnosis'],
        's1_prob':    s1_raw[prob_col_s1],
        's1_pred':    np.where(s1_raw[prob_col_s1] >= s1_th, 'IAS', 'DAS'),
    })

    # ── Load S2 ──
    s2_raw = load_calibrated_probs(paths, '2', scenario, split)

    s2_info = pd.DataFrame({
        's2_true':       s2_raw['target_label'].values,
        's2_pred':       s2_raw['pred_label'].values,
        's2_confidence': s2_raw['confidence'].values,
    })
    # Zone assignment
    s2_info['s2_zone'] = np.where(
        s2_info['s2_confidence'] >= s2_high, 'HIGH',
        np.where(s2_info['s2_confidence'] < s2_low, 'LOW', 'MEDIUM'))

    # ── Positional alignment: attach S2 info to S1-IAS rows ──
    ias_mask = (e2e['s1_true'] == 'IAS')
    for col in s2_info.columns:
        e2e[col] = np.nan
        e2e.loc[ias_mask, col] = s2_info[col].values

    # ── 5-class E2E labels ──
    # True: DAS patients → "DAS", IAS patients → true S2 class
    e2e['e2e_true'] = np.where(e2e['s1_true'] == 'DAS', 'DAS', e2e['s2_true'])

    # Prediction: S1-DAS → "DAS", S1-IAS → S2 prediction
    e2e['e2e_pred'] = np.where(e2e['s1_pred'] == 'DAS', 'DAS', e2e['s2_pred'])

    # E2E zone: S1-DAS → "Excluded", S1-IAS → S2 zone
    e2e['e2e_zone'] = np.where(e2e['s1_pred'] == 'DAS', 'Excluded', e2e['s2_zone'])

    # Correctness & lost-patient flag
    e2e['e2e_correct'] = (e2e['e2e_true'] == e2e['e2e_pred'])
    e2e['is_lost'] = (e2e['s1_true'] == 'IAS') & (e2e['s1_pred'] == 'DAS')

    # Meta
    e2e['scenario'] = scenario
    e2e['split'] = split

    return e2e

# ── All scenarios & splits ──
e2e_results = {}
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        e2e_results[key] = simulate_e2e_pipeline(PATHS, sc, sp, th_config)

# ── Summary ──
print("=" * 65)
print("E2E Pipeline Simulation Summary")
print("=" * 65)

for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        n_total    = len(df)
        n_ias_true = (df['s1_true'] == 'IAS').sum()
        n_das_true = (df['s1_true'] == 'DAS').sum()
        n_s1_ias   = (df['s1_pred'] == 'IAS').sum()
        n_s1_das   = (df['s1_pred'] == 'DAS').sum()
        n_lost     = df['is_lost'].sum()
        n_extra    = ((df['s1_true'] == 'DAS') & (df['s1_pred'] == 'IAS')).sum()
        e2e_acc    = df['e2e_correct'].mean()

        print(f"\n{sc} / {sp.upper()}:")
        print(f"  Total: {n_total} (AAC={n_ias_true}, OAC={n_das_true})")
        print(f"  S1 decision: AAC pred={n_s1_ias}, OAC pred={n_s1_das}")
        print(f"  Lost patients (AAC→OAC): {n_lost}")
        print(f"  Extra      (OAC→AAC): {n_extra}")
        print(f"  E2E Accuracy:          {e2e_acc:.3f}")

## CELL 2 — Load S2 model & predict on extra patients

In [ ]:
!pip install autogluon

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 2 — Load S2 model & predict on extra patients
# ══════════════════════════════════════════════════════════
import joblib
from autogluon.tabular import TabularPredictor

# Find model directories
s2_model_dirs = {}
for sc in SCENARIOS:
    pattern = f"*Stage2_{sc}*"
    s2_dir = list(PATHS.frozen_models.glob(pattern))[0]
    ag_path = s2_dir / 'AutoGluon_Model'
    s2_model_dirs[sc] = ag_path
    print(f"{sc}: {ag_path.name} exists={ag_path.exists()}")

# Load feature names & class mapping
s2_meta = {}
for sc in SCENARIOS:
    s2_dir = list(PATHS.frozen_models.glob(f"*Stage2_{sc}*"))[0]
    s2_meta[sc] = {
        'features': joblib.load(s2_dir / 'feature_names.joblib'),
        'class_map': joblib.load(s2_dir / 'class_mapping.joblib'),
    }
    print(f"{sc}: {len(s2_meta[sc]['features'])} features")
    print(f"  class_names: {s2_meta[sc]['class_map']['class_names']}")

##  CELL 2b — S2 inference on extra patients (v3 — inf fix)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 2b — S2 inference on extra patients (v3 — inf fix)
# ══════════════════════════════════════════════════════════

def compute_missing_ratios(df, feat_names):
    """Compute the ratio features S2 needs from the S1 base."""
    df = df.copy()
    for f in feat_names:
        if f not in df.columns and '_div_' in f:
            num, den = f.split('_div_')
            if num in df.columns and den in df.columns:
                df[f] = df[num] / df[den].replace(0, np.nan)
    # Clean inf and NaN values
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

for sc in SCENARIOS:
    ag_path = s2_model_dirs[sc]
    s2_model = TabularPredictor.load(str(ag_path))
    feat_names = s2_meta[sc]['features']
    class_names = s2_meta[sc]['class_map']['class_names']

    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]

        extra_mask = (df['s1_true'] == 'DAS') & (df['s1_pred'] == 'IAS')
        n_extra = extra_mask.sum()
        if n_extra == 0:
            continue

        s1_raw = load_calibrated_probs(PATHS, '1', sc, sp)
        extra_idx = df.loc[extra_mask, 'idx_s1'].values
        extra_features = compute_missing_ratios(s1_raw.loc[extra_idx], feat_names)

        # NaN/inf check
        n_inf = np.isinf(extra_features[feat_names].select_dtypes(include='number')).sum().sum()
        n_nan = extra_features[feat_names].isna().sum().sum()
        print(f"{sc}/{sp}: inf={n_inf}, NaN={n_nan} (after cleanup)")

        extra_features = extra_features[feat_names].copy()

        # S2 predict
        s2_pred_raw = s2_model.predict(extra_features)
        s2_proba = s2_model.predict_proba(extra_features)

        s2_pred_labels = s2_pred_raw.map(class_names).values
        s2_conf = s2_proba.max(axis=1).values

        # Zone assignment
        op = get_operating_point(th_config, sc)
        s2_high, s2_low = op['s2_zone_high'], op['s2_zone_low']
        zones = np.where(s2_conf >= s2_high, 'HIGH',
                np.where(s2_conf < s2_low, 'LOW', 'MEDIUM'))

        # Update E2E
        e2e_results[key].loc[extra_mask, 's2_pred'] = s2_pred_labels
        e2e_results[key].loc[extra_mask, 's2_confidence'] = s2_conf
        e2e_results[key].loc[extra_mask, 's2_zone'] = zones
        e2e_results[key].loc[extra_mask, 'e2e_pred'] = s2_pred_labels
        e2e_results[key].loc[extra_mask, 'e2e_zone'] = zones
        e2e_results[key].loc[extra_mask, 'e2e_correct'] = (
            e2e_results[key].loc[extra_mask, 'e2e_true'] ==
            e2e_results[key].loc[extra_mask, 'e2e_pred'])

        print(f"✅ {sc}/{sp}: S2 predicted for {n_extra} extra patients")
        _disp = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}
        dist = {_disp.get(k, k): v
                for k, v in pd.Series(s2_pred_labels).value_counts().to_dict().items()}
        print(f"   Distribution: {dist}")

# ── Final kontrol ──
print("\n--- Final check ---")
for key, df in e2e_results.items():
    n_nan = df['e2e_pred'].isna().sum()
    e2e_acc = df['e2e_correct'].mean()
    print(f"{key}: NaN={n_nan}, E2E acc={e2e_acc:.3f}")

## CELL 3 — 5×5 E2E Confusion Matrix (4B)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 3 — 5×5 E2E Confusion Matrix & Figure (4B)
# ══════════════════════════════════════════════════════════
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

E2E_CLASSES = ['DAS', 'DEA', 'HA', 'HGB HTZ', 'Normal']
E2E_DISPLAY = ['OAC', 'IDA', 'HA', 'HGB HTZ', 'Normal']

def e2e_confusion_matrix(df, labels=E2E_CLASSES):
    """Return 5×5 raw + normalized CM."""
    cm_raw = confusion_matrix(df['e2e_true'], df['e2e_pred'], labels=labels)
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)
    return cm_raw, cm_norm

def plot_e2e_cm(cm_raw, cm_norm, labels, ax, title):
    """Single panel: normalized heatmap + raw count annotation."""
    im = ax.imshow(cm_norm, cmap=cmap_tufte, vmin=0, vmax=1, aspect='equal')

    for i in range(len(labels)):
        for j in range(len(labels)):
            raw_val = cm_raw[i, j]
            norm_val = cm_norm[i, j]
            color = 'white' if norm_val > 0.6 else 'black'
            ax.text(j, i, f"{raw_val}\n({norm_val:.2f})",
                    ha='center', va='center', fontsize=7, color=color)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(E2E_DISPLAY, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(E2E_DISPLAY, fontsize=8)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    return im

# ── Compute CM for OOF + Test ──
cms = {}
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        cm_raw, cm_norm = e2e_confusion_matrix(df)
        cms[key] = (cm_raw, cm_norm)

        acc = accuracy_score(df['e2e_true'], df['e2e_pred'])
        f1 = f1_score(df['e2e_true'], df['e2e_pred'], labels=E2E_CLASSES, average='macro')
        print(f"{key}: E2E Accuracy={acc:.3f}, Macro F1={f1:.3f}")

# ── Figure: 1×2 panel (CBC_Only test, CBC_BIO test) ──
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for idx, sc in enumerate(SCENARIOS):
    key = f"{sc}_test"
    cm_raw, cm_norm = cms[key]
    acc = accuracy_score(e2e_results[key]['e2e_true'], e2e_results[key]['e2e_pred'])
    f1 = f1_score(e2e_results[key]['e2e_true'], e2e_results[key]['e2e_pred'],
                  labels=E2E_CLASSES, average='macro')
    im = plot_e2e_cm(cm_raw, cm_norm, E2E_CLASSES, axes[idx],
                     f"{sc}\nAcc={acc:.3f}  F1={f1:.3f}")
    despine_all(axes[idx])

fig.suptitle('E2E Pipeline — 5×5 Confusion Matrix (Test)', fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()

# Colorbar
cbar = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label('Recall (row-normalized)', fontsize=8)

save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'e2e_confusion_matrix_test')
plt.show()
print("✅ Figure saved")

## CELL 4 — Lost Patient Analysis (4C)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 4 — Lost Patient Analysis (4C)
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Lost Patient Analysis — IAS patients missed at S1")
print("=" * 65)

lost_details = {}

for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        lost = df[df['is_lost']].copy()
        lost_details[key] = lost

        n_lost = len(lost)
        n_ias = (df['s1_true'] == 'IAS').sum()

        print(f"\n{'─'*50}")
        print(f"{sc} / {sp.upper()}: {n_lost} lost / {n_ias} IAS ({n_lost/n_ias*100:.1f}%)")
        print(f"{'─'*50}")

        if n_lost == 0:
            continue

        # True S2 class distribution
        print("\nTrue S2 classes (missed):")
        _disp = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}
        dist = lost['s2_true'].value_counts()
        for cls in E2E_CLASSES[1:]:  # exclude DAS
            n = dist.get(cls, 0)
            pct = n / n_lost * 100 if n_lost > 0 else 0
            print(f"  {_disp.get(cls, cls):10s}: {n:3d} ({pct:5.1f}%)")

        # S1 probability statistics
        print(f"\nS1 prob (IAS) — missed:")
        print(f"  Mean:   {lost['s1_prob'].mean():.3f}")
        print(f"  Median: {lost['s1_prob'].median():.3f}")
        print(f"  Min:    {lost['s1_prob'].min():.3f}")
        print(f"  Max:    {lost['s1_prob'].max():.3f}")

# ── Figure: True S2 class distribution of lost patients (test) ──
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

s2_classes = ['DEA', 'HA', 'HGB HTZ', 'Normal']
s2_display = ['IDA', 'HA', 'HGB HTZ', 'Normal']
colors = [PALETTE['highlight'], PALETTE['base1'], PALETTE['accent1'], PALETTE['accent2']]

for idx, sc in enumerate(SCENARIOS):
    key = f"{sc}_test"
    lost = lost_details[key]

    counts = lost['s2_true'].value_counts().reindex(s2_classes, fill_value=0)

    bars = axes[idx].bar(s2_display, counts.values, color=colors, edgecolor='white', width=0.6)

    for bar, val in zip(bars, counts.values):
        if val > 0:
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                          str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

    axes[idx].set_title(f"{sc}\n{len(lost)} lost patients", fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Patient count' if idx == 0 else '', fontsize=9)
    axes[idx].set_xlabel('True S2 class', fontsize=9)
    despine_all(axes[idx])

fig.suptitle('Lost Patients — True Classes of OAC Cases Missed at Tier 1 (Test)',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'lost_patients_distribution')
plt.show()
print("\n✅ Figure saved")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 5 — S1 Threshold Sensitivity Analysis (4C)
# ══════════════════════════════════════════════════════════

thresholds = np.arange(0.10, 0.90, 0.02)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for idx, sc in enumerate(SCENARIOS):
    op = get_operating_point(th_config, sc)
    actual_th = op['s1_threshold']

    lost_counts = []
    extra_counts = []
    e2e_accs = []

    for th in thresholds:
        for sp_check in ['test']:
            key = f"{sc}_{sp_check}"
            df = e2e_results[key].copy()

            # Recompute S1 decision at this threshold
            s1_pred_new = np.where(df['s1_prob'] >= th, 'IAS', 'DAS')

            n_lost = ((df['s1_true'] == 'IAS') & (s1_pred_new == 'DAS')).sum()
            n_extra = ((df['s1_true'] == 'DAS') & (s1_pred_new == 'IAS')).sum()

            lost_counts.append(n_lost)
            extra_counts.append(n_extra)

    ax = axes[idx]
    ax.plot(thresholds, lost_counts, color=PALETTE['highlight'], linewidth=2, label='Lost (AAC→OAC)')
    ax.plot(thresholds, extra_counts, color=PALETTE['base1'], linewidth=1.5,
            linestyle='--', label='Extra (OAC→AAC)')

    # Operating point
    ax.axvline(actual_th, color=PALETTE['accent2'], linewidth=1.5, linestyle=':', alpha=0.8)

    # Values at operating point
    lost_at_op = ((e2e_results[f"{sc}_test"]['s1_true'] == 'IAS') &
                  (e2e_results[f"{sc}_test"]['s1_pred'] == 'DAS')).sum()
    ax.annotate(f'th={actual_th:.2f}\n{lost_at_op} lost',
                xy=(actual_th, lost_at_op), fontsize=8,
                xytext=(actual_th + 0.08, lost_at_op + 5),
                arrowprops=dict(arrowstyle='->', color=PALETTE['accent2'], lw=1.2),
                color=PALETTE['accent2'], fontweight='bold')

    ax.set_xlabel('S1 Threshold', fontsize=9)
    ax.set_ylabel('Patient count' if idx == 0 else '', fontsize=9)
    ax.set_title(f'{sc} (Test)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    despine_all(ax)

fig.suptitle('S1 Threshold Sensitivity — Lost vs Extra Patients',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'threshold_sensitivity')
plt.show()
print("✅ Figure saved")

## CELL 6 — E2E Zone Analysis (4D)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6 — E2E Zone Analysis (4D)
# ══════════════════════════════════════════════════════════

E2E_ZONES = ['Excluded', 'HIGH', 'MEDIUM', 'LOW']
zone_colors = [PALETTE['base2'], PALETTE['accent1'], PALETTE['accent2'], PALETTE['highlight']]

print("=" * 65)
print("E2E Zone Analysis")
print("=" * 65)

zone_summaries = {}

for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]

        print(f"\n{'─'*50}")
        print(f"{sc} / {sp.upper()}")
        print(f"{'─'*50}")

        rows = []
        for zone in E2E_ZONES:
            z_df = df[df['e2e_zone'] == zone]
            n = len(z_df)
            pct = n / len(df) * 100
            acc = z_df['e2e_correct'].mean() if n > 0 else np.nan
            rows.append({'zone': zone, 'n': n, 'pct': pct, 'accuracy': acc})
            print(f"  {zone:10s}: n={n:4d} ({pct:5.1f}%)  acc={acc:.3f}" if n > 0
                  else f"  {zone:10s}: n={n:4d} ({pct:5.1f}%)")

        zone_summaries[key] = pd.DataFrame(rows)

        # Total
        print(f"  {'TOTAL':10s}: n={len(df):4d}          acc={df['e2e_correct'].mean():.3f}")

# ── Figure: Stacked bar — zone distribution (test) ──
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

for idx, sc in enumerate(SCENARIOS):
    key = f"{sc}_test"
    zs = zone_summaries[key]

    bottom = 0
    for i, zone in enumerate(E2E_ZONES):
        row = zs[zs['zone'] == zone].iloc[0]
        bar = axes[idx].bar(0, row['pct'], bottom=bottom, color=zone_colors[i],
                           edgecolor='white', width=0.5, label=zone)

        if row['pct'] > 3:
            acc_str = f"acc={row['accuracy']:.2f}" if not np.isnan(row['accuracy']) else ""
            axes[idx].text(0, bottom + row['pct']/2,
                          f"{zone}\nn={row['n']} ({row['pct']:.1f}%)\n{acc_str}",
                          ha='center', va='center', fontsize=7.5, fontweight='bold')
        bottom += row['pct']

    axes[idx].set_ylim(0, 105)
    axes[idx].set_xlim(-0.5, 0.5)
    axes[idx].set_xticks([])
    axes[idx].set_ylabel('Patient proportion (%)' if idx == 0 else '', fontsize=9)
    axes[idx].set_title(f'{sc}\nE2E Acc={e2e_results[key]["e2e_correct"].mean():.3f}',
                       fontsize=10, fontweight='bold')
    despine_all(axes[idx])

fig.suptitle('E2E Zone Distribution — CDS Operational Output (Test)',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'e2e_zone_distribution')
plt.show()
print("\n✅ Figure saved")

## CELL 7 — E2E Patient Flow Diagram (4E)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7 — E2E Patient Flow Diagram (4E)
# ══════════════════════════════════════════════════════════
# Alluvial-style grouped bar instead of Sankey — clean with matplotlib

# Internal key → display label
E2E_DISP = {'DAS': 'OAC', 'DEA': 'IDA', 'HA': 'HA', 'HGB HTZ': 'HGB HTZ', 'Normal': 'Normal'}

def plot_patient_flow(df, ax, title):
    """Show S1 decision → E2E outcome flow."""

    # S1 layer: IAS vs DAS (pred)
    s1_groups = {'S1→IAS': df[df['s1_pred'] == 'IAS'],
                 'S1→DAS': df[df['s1_pred'] == 'DAS']}

    # E2E outcome classes
    e2e_classes = ['DAS', 'DEA', 'HA', 'HGB HTZ', 'Normal']
    class_colors = {
        'DAS': PALETTE['base2'], 'DEA': PALETTE['highlight'],
        'HA': PALETTE['base1'], 'HGB HTZ': PALETTE['accent1'],
        'Normal': PALETTE['accent2']
    }

    ias_df = s1_groups['S1→IAS']
    das_df = s1_groups['S1→DAS']

    # Stacked horizontal bar: two rows (S1→AAC, S1→OAC)
    y_positions = [1, 0]
    y_labels = [f"S1→AAC (n={len(ias_df)})", f"S1→OAC (n={len(das_df)})"]

    for y_pos, (grp_name, grp_df) in zip(y_positions, s1_groups.items()):
        left = 0
        for cls in e2e_classes:
            # True label distribution
            n = (grp_df['e2e_true'] == cls).sum()
            if n == 0:
                continue
            bar = ax.barh(y_pos, n, left=left, height=0.5,
                         color=class_colors[cls], edgecolor='white', linewidth=0.5)
            if n >= 3:
                ax.text(left + n/2, y_pos, f"{E2E_DISP[cls]}\n{n}",
                       ha='center', va='center', fontsize=7, fontweight='bold')
            left += n

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel('Patient count (true label)', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    despine_all(ax)

fig, axes = plt.subplots(2, 1, figsize=(10, 5))

for idx, sc in enumerate(SCENARIOS):
    key = f"{sc}_test"
    df = e2e_results[key]
    plot_patient_flow(df, axes[idx], f'{sc} — Patient Flow (Test, n=229)')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PALETTE['base2'], label='OAC'),
                   Patch(facecolor=PALETTE['highlight'], label='IDA'),
                   Patch(facecolor=PALETTE['base1'], label='HA'),
                   Patch(facecolor=PALETTE['accent1'], label='HGB HTZ'),
                   Patch(facecolor=PALETTE['accent2'], label='Normal')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=8,
           frameon=False, ncol=5, bbox_to_anchor=(0.98, 1.02))

fig.suptitle('E2E Patient Flow Diagram — S1 Decision → True Class',
             fontsize=11, fontweight='bold', y=1.05)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'e2e_patient_flow')
plt.show()
print("✅ Figure saved")

## CELL 8 — Excel Exports & E2E Metrics Summary (4E)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8 — Excel Exports & E2E Metrics Summary (4E)
# ══════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_fscore_support

analysis_dir = PATHS.module_dir('m4_e2e', 'analysis')

# ── 1. e2e_confusion_matrix.xlsx ──
with pd.ExcelWriter(analysis_dir / 'e2e_confusion_matrix.xlsx') as w:
    for sc in SCENARIOS:
        for sp in ['oof', 'test']:
            key = f"{sc}_{sp}"
            cm_raw, cm_norm = e2e_confusion_matrix(e2e_results[key])

            df_raw = pd.DataFrame(cm_raw, index=E2E_CLASSES, columns=E2E_CLASSES)
            df_norm = pd.DataFrame(cm_norm.round(4), index=E2E_CLASSES, columns=E2E_CLASSES)

            df_raw.to_excel(w, sheet_name=f"{sc}_{sp}_raw")
            df_norm.to_excel(w, sheet_name=f"{sc}_{sp}_norm")
print("✅ e2e_confusion_matrix.xlsx")

# ── 2. e2e_metrics.xlsx ──
metrics_rows = []
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        y_true, y_pred = df['e2e_true'], df['e2e_pred']

        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, labels=E2E_CLASSES, average='macro')
        p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=E2E_CLASSES)

        for i, cls in enumerate(E2E_CLASSES):
            metrics_rows.append({
                'scenario': sc, 'split': sp, 'class': cls,
                'precision': round(p[i], 4), 'recall': round(r[i], 4),
                'f1': round(f1[i], 4), 'support': int(sup[i]),
                'e2e_accuracy': round(acc, 4), 'e2e_f1_macro': round(f1_macro, 4)
            })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_excel(analysis_dir / 'e2e_metrics.xlsx', index=False)
print("✅ e2e_metrics.xlsx")

# ── 3. lost_patients.xlsx ──
lost_rows = []
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        lost = e2e_results[key][e2e_results[key]['is_lost']].copy()
        lost_rows.append(lost)

lost_all = pd.concat(lost_rows, ignore_index=True)
lost_all.to_excel(analysis_dir / 'lost_patients.xlsx', index=False)
print("✅ lost_patients.xlsx")

# ── 4. e2e_zone_summary.xlsx ──
zone_rows = []
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        for zone in E2E_ZONES:
            z_df = df[df['e2e_zone'] == zone]
            zone_rows.append({
                'scenario': sc, 'split': sp, 'zone': zone,
                'n': len(z_df), 'pct': round(len(z_df)/len(df)*100, 1),
                'accuracy': round(z_df['e2e_correct'].mean(), 4) if len(z_df) > 0 else np.nan
            })

zone_df = pd.DataFrame(zone_rows)
zone_df.to_excel(analysis_dir / 'e2e_zone_summary.xlsx', index=False)
print("✅ e2e_zone_summary.xlsx")

# ── 5. e2e_scenario_comparison.xlsx ──
comp_rows = []
for sc in SCENARIOS:
    for sp in ['oof', 'test']:
        key = f"{sc}_{sp}"
        df = e2e_results[key]
        acc = accuracy_score(df['e2e_true'], df['e2e_pred'])
        f1m = f1_score(df['e2e_true'], df['e2e_pred'], labels=E2E_CLASSES, average='macro')
        n_lost = df['is_lost'].sum()
        n_extra = ((df['s1_true'] == 'DAS') & (df['s1_pred'] == 'IAS')).sum()
        high_pct = (df['e2e_zone'] == 'HIGH').sum() / len(df) * 100
        high_acc = df[df['e2e_zone'] == 'HIGH']['e2e_correct'].mean() if (df['e2e_zone'] == 'HIGH').any() else np.nan

        comp_rows.append({
            'scenario': sc, 'split': sp,
            'e2e_accuracy': round(acc, 4), 'e2e_f1_macro': round(f1m, 4),
            'lost_patients': n_lost, 'extra_patients': n_extra,
            'high_zone_pct': round(high_pct, 1), 'high_zone_acc': round(high_acc, 4) if not np.isnan(high_acc) else np.nan
        })

comp_df = pd.DataFrame(comp_rows)
comp_df.to_excel(analysis_dir / 'e2e_scenario_comparison.xlsx', index=False)
print("✅ e2e_scenario_comparison.xlsx")

# ── Print summary table ──
print("\n" + "=" * 65)
print("E2E Scenario Comparison (Test)")
print("=" * 65)
print(comp_df[comp_df['split'] == 'test'].to_string(index=False))

## CELL 9 — Create src/m4_e2e.py

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9 — Create src/m4_e2e.py
# ══════════════════════════════════════════════════════════

m4_code = '''"""
M4 — End-to-End Pipeline Analysis
Reusable functions for E2E simulation, confusion matrix, lost patient analysis.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_recall_fscore_support

from m1_calibration import load_calibrated_probs
from m2_thresholds import load_threshold_config, get_operating_point


E2E_CLASSES = ['DAS', 'DEA', 'HA', 'HGB HTZ', 'Normal']
E2E_ZONES   = ['Excluded', 'HIGH', 'MEDIUM', 'LOW']


def simulate_e2e_pipeline(paths, scenario, split, th_config):
    """
    E2E pipeline simulation — S1-S2 merge via positional alignment.
    S1 filter → routing to S2 → 5-class E2E label.

    Returns: DataFrame with columns:
        idx_s1, s1_true, diagnosis, s1_prob, s1_pred,
        s2_true, s2_pred, s2_confidence, s2_zone,
        e2e_true, e2e_pred, e2e_zone, e2e_correct, is_lost,
        scenario, split
    """
    op = get_operating_point(th_config, scenario)
    s1_th  = op['s1_threshold']
    s2_low = op['s2_zone_low']
    s2_high = op['s2_zone_high']

    # Load S1
    s1_raw = load_calibrated_probs(paths, '1', scenario, split)
    prob_col_s1 = 'prob_IAS_cal' if 'prob_IAS_cal' in s1_raw.columns else 'prob_IAS'

    e2e = pd.DataFrame({
        'idx_s1':     s1_raw.index,
        's1_true':    s1_raw['target'].map({1: 'IAS', 0: 'DAS'}),
        'diagnosis':  s1_raw['diagnosis'],
        's1_prob':    s1_raw[prob_col_s1],
        's1_pred':    np.where(s1_raw[prob_col_s1] >= s1_th, 'IAS', 'DAS'),
    })

    # Load S2
    s2_raw = load_calibrated_probs(paths, '2', scenario, split)
    s2_info = pd.DataFrame({
        's2_true':       s2_raw['target_label'].values,
        's2_pred':       s2_raw['pred_label'].values,
        's2_confidence': s2_raw['confidence'].values,
    })
    s2_info['s2_zone'] = np.where(
        s2_info['s2_confidence'] >= s2_high, 'HIGH',
        np.where(s2_info['s2_confidence'] < s2_low, 'LOW', 'MEDIUM'))

    # Positional alignment: attach S2 info to S1-IAS rows
    ias_mask = (e2e['s1_true'] == 'IAS')
    for col in s2_info.columns:
        e2e[col] = np.nan
        e2e.loc[ias_mask, col] = s2_info[col].values

    # 5-class E2E labels
    e2e['e2e_true'] = np.where(e2e['s1_true'] == 'DAS', 'DAS', e2e['s2_true'])
    e2e['e2e_pred'] = np.where(e2e['s1_pred'] == 'DAS', 'DAS', e2e['s2_pred'])
    e2e['e2e_zone'] = np.where(e2e['s1_pred'] == 'DAS', 'Excluded', e2e['s2_zone'])
    e2e['e2e_correct'] = (e2e['e2e_true'] == e2e['e2e_pred'])
    e2e['is_lost'] = (e2e['s1_true'] == 'IAS') & (e2e['s1_pred'] == 'DAS')
    e2e['scenario'] = scenario
    e2e['split'] = split

    return e2e


def e2e_confusion_matrix(df, labels=None):
    """Return 5×5 raw + row-normalized confusion matrix."""
    if labels is None:
        labels = E2E_CLASSES
    cm_raw = confusion_matrix(df['e2e_true'], df['e2e_pred'], labels=labels)
    row_sums = cm_raw.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_raw.astype(float), row_sums,
                        where=row_sums != 0, out=np.zeros_like(cm_raw, dtype=float))
    return cm_raw, cm_norm


def e2e_metrics(df, labels=None):
    """E2E accuracy, macro F1, per-class precision/recall/f1."""
    if labels is None:
        labels = E2E_CLASSES
    y_true, y_pred = df['e2e_true'], df['e2e_pred']
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, labels=labels, average='macro')
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels)
    per_class = {cls: {'precision': p[i], 'recall': r[i], 'f1': f1[i], 'support': int(sup[i])}
                 for i, cls in enumerate(labels)}
    return {'accuracy': acc, 'f1_macro': f1m, 'per_class': per_class}


def lost_patient_analysis(df):
    """Summary statistics for lost patients (is_lost=True)."""
    lost = df[df['is_lost']].copy()
    n_ias = (df['s1_true'] == 'IAS').sum()

    summary = {
        'n_lost': len(lost),
        'n_ias': n_ias,
        'lost_rate': len(lost) / n_ias if n_ias > 0 else 0,
        's2_true_distribution': lost['s2_true'].value_counts().to_dict() if len(lost) > 0 else {},
        's1_prob_stats': {
            'mean': lost['s1_prob'].mean(),
            'median': lost['s1_prob'].median(),
            'min': lost['s1_prob'].min(),
            'max': lost['s1_prob'].max(),
        } if len(lost) > 0 else {},
    }
    return summary


def e2e_zone_summary(df):
    """Per-zone E2E statistics."""
    rows = []
    for zone in E2E_ZONES:
        z_df = df[df['e2e_zone'] == zone]
        rows.append({
            'zone': zone, 'n': len(z_df),
            'pct': len(z_df) / len(df) * 100,
            'accuracy': z_df['e2e_correct'].mean() if len(z_df) > 0 else np.nan
        })
    return pd.DataFrame(rows)


def compute_missing_ratios(df, feat_names):
    """Compute S2 ratio features from S1 base features."""
    df = df.copy()
    for f in feat_names:
        if f not in df.columns and '_div_' in f:
            num, den = f.split('_div_')
            if num in df.columns and den in df.columns:
                df[f] = df[num] / df[den].replace(0, np.nan)
    df = df.replace([np.inf, -np.inf], np.nan)
    return df
'''

src_path = PATHS.src / 'm4_e2e.py'
with open(src_path, 'w') as f:
    f.write(m4_code)

print(f"✅ {src_path} written ({len(m4_code)} characters)")

## CELL 10 — Manifest Update & M4 Summary

## CELL S11a — E2E-Optimal S1 Threshold Search

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1a — E2E-Optimal S1 Threshold Search
# ══════════════════════════════════════════════════════════

from sklearn.metrics import f1_score, accuracy_score

thresholds = np.arange(0.10, 0.90, 0.01)

def evaluate_e2e_at_threshold(e2e_df, new_th):
    """Recompute E2E metrics after changing the S1 threshold on the current E2E DataFrame."""
    df = e2e_df.copy()

    df['s1_pred_new'] = np.where(df['s1_prob'] >= new_th, 'IAS', 'DAS')
    df['e2e_pred_new'] = np.where(df['s1_pred_new'] == 'DAS', 'DAS', df['s2_pred'])

    valid = df['e2e_pred_new'].notna()

    if valid.sum() < len(df) * 0.5:
        return None

    acc = accuracy_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_new'])
    f1  = f1_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_new'],
                   labels=E2E_CLASSES, average='macro')
    n_lost  = ((df['s1_true'] == 'IAS') & (df['s1_pred_new'] == 'DAS')).sum()
    n_extra = ((df['s1_true'] == 'DAS') & (df['s1_pred_new'] == 'IAS')).sum()
    n_nan   = (~valid).sum()

    return {'threshold': new_th, 'e2e_acc': acc, 'e2e_f1': f1,
            'n_lost': n_lost, 'n_extra': n_extra, 'n_nan': n_nan}

# ── Threshold sweep per scenario ──
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, sc in enumerate(SCENARIOS):
    oof_key = f"{sc}_oof"
    test_key = f"{sc}_test"

    oof_results = []
    test_results = []

    for th in thresholds:
        r_oof = evaluate_e2e_at_threshold(e2e_results[oof_key], th)
        r_test = evaluate_e2e_at_threshold(e2e_results[test_key], th)
        if r_oof: oof_results.append(r_oof)
        if r_test: test_results.append(r_test)

    oof_df = pd.DataFrame(oof_results)
    test_df = pd.DataFrame(test_results)

    best_oof_acc = oof_df.loc[oof_df['e2e_acc'].idxmax()]
    best_oof_f1  = oof_df.loc[oof_df['e2e_f1'].idxmax()]

    op = get_operating_point(th_config, sc)
    current_th = op['s1_threshold']

    print(f"\n{'='*55}")
    print(f"{sc}")
    print(f"{'='*55}")
    print(f"Current threshold:     {current_th:.2f}")
    print(f"OOF E2E-Acc optimal:   th={best_oof_acc['threshold']:.2f}  "
          f"acc={best_oof_acc['e2e_acc']:.3f}  f1={best_oof_acc['e2e_f1']:.3f}  "
          f"lost={int(best_oof_acc['n_lost'])}  extra={int(best_oof_acc['n_extra'])}")
    print(f"OOF E2E-F1 optimal:    th={best_oof_f1['threshold']:.2f}  "
          f"acc={best_oof_f1['e2e_acc']:.3f}  f1={best_oof_f1['e2e_f1']:.3f}  "
          f"lost={int(best_oof_f1['n_lost'])}  extra={int(best_oof_f1['n_extra'])}")

    for label, best_th in [('E2E-Acc', best_oof_acc['threshold']),
                            ('E2E-F1', best_oof_f1['threshold']),
                            ('Current', current_th)]:
        t_row = test_df[test_df['threshold'] == round(best_th, 2)]
        if len(t_row) > 0:
            t = t_row.iloc[0]
            print(f"  → Test ({label}):  acc={t['e2e_acc']:.3f}  f1={t['e2e_f1']:.3f}  "
                  f"lost={int(t['n_lost'])}  extra={int(t['n_extra'])}")

    # ── Plot: Accuracy & F1 vs threshold ──
    ax1 = axes[idx, 0]
    ax1.plot(oof_df['threshold'], oof_df['e2e_acc'], color=PALETTE['base1'],
             linewidth=1.5, label='OOF Acc')
    ax1.plot(test_df['threshold'], test_df['e2e_acc'], color=PALETTE['base1'],
             linewidth=1.5, linestyle='--', label='Test Acc')
    ax1.plot(oof_df['threshold'], oof_df['e2e_f1'], color=PALETTE['highlight'],
             linewidth=1.5, label='OOF F1')
    ax1.plot(test_df['threshold'], test_df['e2e_f1'], color=PALETTE['highlight'],
             linewidth=1.5, linestyle='--', label='Test F1')
    ax1.axvline(current_th, color=PALETTE['accent2'], linestyle=':', linewidth=1.5, label=f'Current ({current_th})')
    ax1.axvline(best_oof_acc['threshold'], color=PALETTE['accent1'], linestyle=':',
                linewidth=1.5, label=f'E2E-opt ({best_oof_acc["threshold"]:.2f})')
    ax1.set_xlabel('S1 Threshold', fontsize=9)
    ax1.set_ylabel('E2E Metric', fontsize=9)
    ax1.set_title(f'{sc} — E2E Metrics vs Threshold', fontsize=10, fontweight='bold')
    ax1.legend(fontsize=7, frameon=False)
    despine_all(ax1)

    # ── Plot: Lost vs Extra ──
    ax2 = axes[idx, 1]
    ax2.plot(oof_df['threshold'], oof_df['n_lost'], color=PALETTE['highlight'],
             linewidth=1.5, label='Lost (OOF)')
    ax2.plot(oof_df['threshold'], oof_df['n_extra'], color=PALETTE['base1'],
             linewidth=1.5, linestyle='--', label='Extra (OOF)')
    ax2.axvline(current_th, color=PALETTE['accent2'], linestyle=':', linewidth=1.5)
    ax2.axvline(best_oof_acc['threshold'], color=PALETTE['accent1'], linestyle=':', linewidth=1.5)
    ax2.set_xlabel('S1 Threshold', fontsize=9)
    ax2.set_ylabel('Patient count', fontsize=9)
    ax2.set_title(f'{sc} — Lost vs Extra', fontsize=10, fontweight='bold')
    ax2.legend(fontsize=7, frameon=False)
    despine_all(ax2)

fig.suptitle('Strategy 1a — E2E-Optimal S1 Threshold Search',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'strategy_1a_threshold_search')
plt.show()

# CELL S11a-verify — Test verification + 1a summary

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1a-verify — Test verification + 1a summary
# ══════════════════════════════════════════════════════════

# Current vs E2E-optimal comparison on test
print("=" * 65)
print("Strategy 1a — Test Verification")
print("=" * 65)

s1a_results = {}

for sc in SCENARIOS:
    test_key = f"{sc}_test"
    op = get_operating_point(th_config, sc)
    current_th = op['s1_threshold']

    # E2E-optimal threshold (from OOF)
    e2e_opt_th = 0.26 if sc == 'CBC_Only' else 0.50

    print(f"\n{sc}:")
    for label, th in [('Current', current_th), ('E2E-opt', e2e_opt_th)]:
        r = evaluate_e2e_at_threshold(e2e_results[test_key], th)
        if r:
            print(f"  {label:10s} (th={th:.2f}): acc={r['e2e_acc']:.3f}  f1={r['e2e_f1']:.3f}  "
                  f"lost={r['n_lost']}  extra={r['n_extra']}  nan={r['n_nan']}")
            s1a_results[f"{sc}_{label}"] = r

## CELL S1b — DAS Rejection Mechanism

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1b — DAS Rejection Mechanism
# ══════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("Strategy 1b — S2 LOW Confidence → OAC Rejection")
print("=" * 65)

def apply_das_rejection(e2e_df, rejection_threshold):
    """S1→IAS but S2 confidence < rejection_threshold → revert to DAS."""
    df = e2e_df.copy()

    # Patients with S1→IAS and low S2 confidence
    reject_mask = ((df['s1_pred'] == 'IAS') &
                   (df['s2_confidence'].astype(float) < rejection_threshold))

    # Set e2e_pred to DAS for these patients
    df.loc[reject_mask, 'e2e_pred_1b'] = 'DAS'
    df.loc[~reject_mask, 'e2e_pred_1b'] = df.loc[~reject_mask, 'e2e_pred']

    valid = df['e2e_pred_1b'].notna()
    acc = accuracy_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_1b'])
    f1 = f1_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_1b'],
                  labels=E2E_CLASSES, average='macro')
    n_rejected = reject_mask.sum()

    # Rejection accuracy — how many rejected are truly DAS?
    rejected_correct = (df.loc[reject_mask, 'e2e_true'] == 'DAS').sum() if n_rejected > 0 else 0

    return {'rej_th': rejection_threshold, 'e2e_acc': acc, 'e2e_f1': f1,
            'n_rejected': n_rejected, 'rejected_truly_das': rejected_correct,
            'rejection_precision': rejected_correct / n_rejected if n_rejected > 0 else 0}

rej_thresholds = np.arange(0.25, 0.65, 0.01)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for idx, sc in enumerate(SCENARIOS):
    oof_rej = [apply_das_rejection(e2e_results[f"{sc}_oof"], rt) for rt in rej_thresholds]
    test_rej = [apply_das_rejection(e2e_results[f"{sc}_test"], rt) for rt in rej_thresholds]

    oof_rej_df = pd.DataFrame(oof_rej)
    test_rej_df = pd.DataFrame(test_rej)

    # Baseline (no rejection)
    base_acc_oof = e2e_results[f"{sc}_oof"]['e2e_correct'].mean()
    base_acc_test = e2e_results[f"{sc}_test"]['e2e_correct'].mean()

    # Best rejection threshold on OOF
    best_oof = oof_rej_df.loc[oof_rej_df['e2e_acc'].idxmax()]

    print(f"\n{sc}:")
    print(f"  Baseline (no rejection):  OOF acc={base_acc_oof:.3f}  Test acc={base_acc_test:.3f}")
    print(f"  Best rejection (OOF):     rej_th={best_oof['rej_th']:.2f}  acc={best_oof['e2e_acc']:.3f}  "
          f"f1={best_oof['e2e_f1']:.3f}  rejected={int(best_oof['n_rejected'])}  "
          f"rej_precision={best_oof['rejection_precision']:.2f}")

    # This threshold on test
    test_at_best = test_rej_df.loc[(test_rej_df['rej_th'] - best_oof['rej_th']).abs().idxmin()]
    print(f"  Test (at best OOF th):    acc={test_at_best['e2e_acc']:.3f}  "
          f"f1={test_at_best['e2e_f1']:.3f}  rejected={int(test_at_best['n_rejected'])}  "
          f"rej_precision={test_at_best['rejection_precision']:.2f}")

    s1a_results[f"{sc}_1b_baseline"] = {'e2e_acc': base_acc_test}
    s1a_results[f"{sc}_1b_best"] = {'e2e_acc': test_at_best['e2e_acc'],
                                     'rej_th': best_oof['rej_th']}

    # Plot
    ax = axes[idx]
    ax.plot(oof_rej_df['rej_th'], oof_rej_df['e2e_acc'], color=PALETTE['base1'],
            linewidth=1.5, label='OOF Acc')
    ax.plot(test_rej_df['rej_th'], test_rej_df['e2e_acc'], color=PALETTE['base1'],
            linewidth=1.5, linestyle='--', label='Test Acc')
    ax.plot(oof_rej_df['rej_th'], oof_rej_df['e2e_f1'], color=PALETTE['highlight'],
            linewidth=1.5, label='OOF F1')
    ax.plot(test_rej_df['rej_th'], test_rej_df['e2e_f1'], color=PALETTE['highlight'],
            linewidth=1.5, linestyle='--', label='Test F1')
    ax.axhline(base_acc_oof, color=PALETTE['accent2'], linestyle=':', linewidth=1, alpha=0.7, label='Baseline')
    ax.axvline(best_oof['rej_th'], color=PALETTE['accent1'], linestyle=':', linewidth=1.5)
    ax.set_xlabel('S2 Rejection Threshold', fontsize=9)
    ax.set_ylabel('E2E Metric', fontsize=9)
    ax.set_title(f'{sc} — OAC Rejection', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, frameon=False)
    despine_all(ax)

fig.suptitle('Strategy 1b — S2 LOW Confidence → OAC Rejection',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'strategy_1b_das_rejection')
plt.show()

## CELL S1c — E2E Confidence Score (S1 prob × S2 confidence)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1c — E2E Confidence Score (S1 prob × S2 confidence)
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Strategy 1c — E2E Confidence Score")
print("=" * 65)

def apply_e2e_confidence(e2e_df, e2e_threshold, method='product'):
    """
    Combine S1 prob and S2 confidence into an E2E confidence score.
    Mark low-confidence patients as "Uncertain" → revert to DAS.
    """
    df = e2e_df.copy()

    # Compute E2E confidence only for S1→IAS patients
    ias_mask = (df['s1_pred'] == 'IAS')

    s2_conf = df['s2_confidence'].astype(float)

    if method == 'product':
        df.loc[ias_mask, 'e2e_conf'] = df.loc[ias_mask, 's1_prob'] * s2_conf[ias_mask]
    elif method == 'min':
        df.loc[ias_mask, 'e2e_conf'] = np.minimum(df.loc[ias_mask, 's1_prob'], s2_conf[ias_mask])
    elif method == 'harmonic':
        s1p = df.loc[ias_mask, 's1_prob']
        s2c = s2_conf[ias_mask]
        df.loc[ias_mask, 'e2e_conf'] = 2 * s1p * s2c / (s1p + s2c)

    # S1→DAS patients: e2e_conf = 1 - s1_prob (DAS confidence)
    das_mask = (df['s1_pred'] == 'DAS')
    df.loc[das_mask, 'e2e_conf'] = 1 - df.loc[das_mask, 's1_prob']

    # Low E2E confidence S1→IAS patients → reject to DAS
    reject_mask = ias_mask & (df['e2e_conf'] < e2e_threshold)

    df['e2e_pred_1c'] = df['e2e_pred']
    df.loc[reject_mask, 'e2e_pred_1c'] = 'DAS'

    valid = df['e2e_pred_1c'].notna()
    acc = accuracy_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_1c'])
    f1 = f1_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_1c'],
                  labels=E2E_CLASSES, average='macro')
    n_rejected = reject_mask.sum()
    rejected_truly_das = (df.loc[reject_mask, 'e2e_true'] == 'DAS').sum() if n_rejected > 0 else 0

    return {'e2e_th': e2e_threshold, 'method': method,
            'e2e_acc': acc, 'e2e_f1': f1,
            'n_rejected': n_rejected, 'rej_truly_das': rejected_truly_das,
            'rej_precision': rejected_truly_das / n_rejected if n_rejected > 0 else 0}

# ── 3 methods × threshold sweep ──
e2e_thresholds = np.arange(0.10, 0.55, 0.01)
methods = ['product', 'min', 'harmonic']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, sc in enumerate(SCENARIOS):
    base_acc_test = e2e_results[f"{sc}_test"]['e2e_correct'].mean()

    print(f"\n{'─'*55}")
    print(f"{sc} (baseline test acc={base_acc_test:.3f})")
    print(f"{'─'*55}")

    best_per_method = {}

    for method in methods:
        oof_res = [apply_e2e_confidence(e2e_results[f"{sc}_oof"], th, method) for th in e2e_thresholds]
        test_res = [apply_e2e_confidence(e2e_results[f"{sc}_test"], th, method) for th in e2e_thresholds]

        oof_df = pd.DataFrame(oof_res)
        test_df = pd.DataFrame(test_res)

        # OOF best
        best_idx = oof_df['e2e_acc'].idxmax()
        best_oof = oof_df.iloc[best_idx]
        best_test = test_df.iloc[best_idx]

        best_per_method[method] = {
            'oof': best_oof, 'test': best_test,
            'oof_curve': oof_df, 'test_curve': test_df
        }

        print(f"  {method:10s}: OOF best th={best_oof['e2e_th']:.2f}  "
              f"acc={best_oof['e2e_acc']:.3f}  rej={int(best_oof['n_rejected'])}  "
              f"rej_prec={best_oof['rej_precision']:.2f}")
        print(f"  {'':10s}  Test:      "
              f"acc={best_test['e2e_acc']:.3f}  rej={int(best_test['n_rejected'])}  "
              f"rej_prec={best_test['rej_precision']:.2f}")

    # Plot — OOF acc by method
    method_colors = {'product': PALETTE['highlight'], 'min': PALETTE['base1'],
                     'harmonic': PALETTE['accent3']}

    ax1 = axes[idx, 0]
    for method in methods:
        curve = best_per_method[method]['oof_curve']
        ax1.plot(curve['e2e_th'], curve['e2e_acc'], color=method_colors[method],
                linewidth=1.5, label=f'{method} (OOF)')
    ax1.axhline(e2e_results[f"{sc}_oof"]['e2e_correct'].mean(),
                color=PALETTE['accent2'], linestyle=':', linewidth=1, label='Baseline')
    ax1.set_xlabel('E2E Confidence Threshold', fontsize=9)
    ax1.set_ylabel('E2E Accuracy (OOF)', fontsize=9)
    ax1.set_title(f'{sc} — OOF', fontsize=10, fontweight='bold')
    ax1.legend(fontsize=7, frameon=False)
    despine_all(ax1)

    # Plot — rejection count
    ax2 = axes[idx, 1]
    for method in methods:
        curve = best_per_method[method]['oof_curve']
        ax2.plot(curve['e2e_th'], curve['n_rejected'], color=method_colors[method],
                linewidth=1.5, label=f'{method}')
    ax2.set_xlabel('E2E Confidence Threshold', fontsize=9)
    ax2.set_ylabel('Rejected Patients (OOF)', fontsize=9)
    ax2.set_title(f'{sc} — Rejection Count', fontsize=10, fontweight='bold')
    ax2.legend(fontsize=7, frameon=False)
    despine_all(ax2)

fig.suptitle('Strategy 1c — E2E Confidence Score (S1×S2 Combination)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'strategy_1c_e2e_confidence')
plt.show()

## CELL S1-COMBO — Combined 1a + 1c & Summary of All Strategies


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1-COMBO — 1a + 1c Combined & All Strategies Summary
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Combined Strategy — 1a (E2E-opt threshold) + 1c (E2E confidence)")
print("=" * 65)

def evaluate_combined(e2e_df, s1_threshold, e2e_conf_threshold, conf_method='product'):
    """Combine 1a (different S1 threshold) + 1c (E2E confidence rejection)."""
    df = e2e_df.copy()

    # 1a: change S1 threshold
    df['s1_pred_new'] = np.where(df['s1_prob'] >= s1_threshold, 'IAS', 'DAS')
    df['e2e_pred_combo'] = np.where(df['s1_pred_new'] == 'DAS', 'DAS', df['s2_pred'])

    # 1c: compute E2E confidence and reject low-confidence
    ias_mask = (df['s1_pred_new'] == 'IAS')
    s2_conf = df['s2_confidence'].astype(float)

    if conf_method == 'product':
        df.loc[ias_mask, 'e2e_conf'] = df.loc[ias_mask, 's1_prob'] * s2_conf[ias_mask]
    elif conf_method == 'harmonic':
        s1p = df.loc[ias_mask, 's1_prob']
        s2c = s2_conf[ias_mask]
        df.loc[ias_mask, 'e2e_conf'] = 2 * s1p * s2c / (s1p + s2c)

    reject_mask = ias_mask & (df['e2e_conf'] < e2e_conf_threshold)
    df.loc[reject_mask, 'e2e_pred_combo'] = 'DAS'

    valid = df['e2e_pred_combo'].notna()
    if valid.sum() < len(df) * 0.5:
        return None

    acc = accuracy_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_combo'])
    f1 = f1_score(df.loc[valid, 'e2e_true'], df.loc[valid, 'e2e_pred_combo'],
                  labels=E2E_CLASSES, average='macro')
    n_lost = ((df['s1_true'] == 'IAS') & (df['s1_pred_new'] == 'DAS')).sum()
    n_rejected = reject_mask.sum()

    return {'s1_th': s1_threshold, 'e2e_conf_th': e2e_conf_threshold,
            'e2e_acc': acc, 'e2e_f1': f1, 'n_lost': n_lost, 'n_rejected': n_rejected}

# ── Grid search (OOF) ──
s1_ths = np.arange(0.20, 0.65, 0.02)
conf_ths = np.arange(0.15, 0.50, 0.02)

for sc in SCENARIOS:
    base_test = e2e_results[f"{sc}_test"]['e2e_correct'].mean()
    base_oof = e2e_results[f"{sc}_oof"]['e2e_correct'].mean()

    # Grid search on OOF
    grid_results = []
    for s1_th in s1_ths:
        for conf_th in conf_ths:
            r = evaluate_combined(e2e_results[f"{sc}_oof"], s1_th, conf_th, 'product')
            if r:
                grid_results.append(r)

    grid_df = pd.DataFrame(grid_results)

    # Top 5 OOF combinations
    top5 = grid_df.nlargest(5, 'e2e_acc')

    print(f"\n{'─'*55}")
    print(f"{sc} — Top 5 OOF Combinations (baseline OOF={base_oof:.3f}, Test={base_test:.3f})")
    print(f"{'─'*55}")

    for _, row in top5.iterrows():
        # Test verification
        t = evaluate_combined(e2e_results[f"{sc}_test"], row['s1_th'], row['e2e_conf_th'], 'product')
        if t:
            print(f"  S1={row['s1_th']:.2f} Conf={row['e2e_conf_th']:.2f} → "
                  f"OOF acc={row['e2e_acc']:.3f} f1={row['e2e_f1']:.3f} | "
                  f"TEST acc={t['e2e_acc']:.3f} f1={t['e2e_f1']:.3f} "
                  f"lost={t['n_lost']} rej={t['n_rejected']}")

# ══════════════════════════════════════════════════════════
# GRAND SUMMARY — Comparison of all strategies
# ══════════════════════════════════════════════════════════

print("\n\n" + "=" * 75)
print("GRAND SUMMARY — Strategy 1 Results (Test Set)")
print("=" * 75)

summary_data = []

for sc in SCENARIOS:
    base = e2e_results[f"{sc}_test"]['e2e_correct'].mean()
    base_f1 = f1_score(e2e_results[f"{sc}_test"]['e2e_true'],
                       e2e_results[f"{sc}_test"]['e2e_pred'],
                       labels=E2E_CLASSES, average='macro')

    summary_data.append({'Scenario': sc, 'Strategy': 'Baseline',
                        'Test Acc': f"{base:.3f}", 'Test F1': f"{base_f1:.3f}", 'Notes': 'M4 original'})

    # 1a
    e2e_opt_th = 0.26 if sc == 'CBC_Only' else 0.50
    r1a = evaluate_e2e_at_threshold(e2e_results[f"{sc}_test"], e2e_opt_th)
    if r1a:
        summary_data.append({'Scenario': sc, 'Strategy': '1a E2E-opt th',
                            'Test Acc': f"{r1a['e2e_acc']:.3f}", 'Test F1': f"{r1a['e2e_f1']:.3f}",
                            'Notes': f"th={e2e_opt_th}, lost={r1a['n_lost']}"})

    # 1b
    summary_data.append({'Scenario': sc, 'Strategy': '1b OAC reject',
                        'Test Acc': f"{base:.3f}", 'Test F1': f"{base_f1:.3f}",
                        'Notes': 'No effect (LOW zone empty)'})

    # 1c (product best)
    r1c = apply_e2e_confidence(e2e_results[f"{sc}_test"],
                                0.33 if sc == 'CBC_Only' else 0.39, 'product')
    summary_data.append({'Scenario': sc, 'Strategy': '1c E2E conf',
                        'Test Acc': f"{r1c['e2e_acc']:.3f}", 'Test F1': f"{r1c['e2e_f1']:.3f}",
                        'Notes': f"rej={r1c['n_rejected']}, prec={r1c['rej_precision']:.2f}"})

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

## CELL S1-SAVE — Strategy 1 Results: Excel + Figure

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S1-SAVE — Strategy 1 Results: Excel + Figure
# ══════════════════════════════════════════════════════════

analysis_dir = PATHS.module_dir('m4_e2e', 'analysis')

# ── 1. Grand Summary Excel ──
rows = []
for sc in SCENARIOS:
    base_df = e2e_results[f"{sc}_test"]
    base_acc = base_df['e2e_correct'].mean()
    base_f1 = f1_score(base_df['e2e_true'], base_df['e2e_pred'],
                       labels=E2E_CLASSES, average='macro')
    n_lost_base = base_df['is_lost'].sum()

    # Baseline
    rows.append({'scenario': sc, 'strategy': 'Baseline (M4)',
                 'test_acc': round(base_acc, 4), 'test_f1': round(base_f1, 4),
                 'delta_acc': 0, 'n_lost': n_lost_base, 'n_rejected': 0,
                 'rejection_precision': np.nan, 'notes': 'Original M4 pipeline'})

    # 1a
    e2e_opt_th = 0.26 if sc == 'CBC_Only' else 0.50
    r1a = evaluate_e2e_at_threshold(base_df, e2e_opt_th)
    rows.append({'scenario': sc, 'strategy': '1a E2E-optimal threshold',
                 'test_acc': round(r1a['e2e_acc'], 4), 'test_f1': round(r1a['e2e_f1'], 4),
                 'delta_acc': round(r1a['e2e_acc'] - base_acc, 4),
                 'n_lost': r1a['n_lost'], 'n_rejected': 0,
                 'rejection_precision': np.nan,
                 'notes': f"S1 th={e2e_opt_th}"})

    # 1b
    rows.append({'scenario': sc, 'strategy': '1b OAC rejection',
                 'test_acc': round(base_acc, 4), 'test_f1': round(base_f1, 4),
                 'delta_acc': 0, 'n_lost': n_lost_base, 'n_rejected': 0,
                 'rejection_precision': np.nan,
                 'notes': 'No effect — LOW zone empty'})

    # 1c (product)
    conf_th = 0.33 if sc == 'CBC_Only' else 0.39
    r1c = apply_e2e_confidence(base_df, conf_th, 'product')
    rows.append({'scenario': sc, 'strategy': '1c E2E confidence (product)',
                 'test_acc': round(r1c['e2e_acc'], 4), 'test_f1': round(r1c['e2e_f1'], 4),
                 'delta_acc': round(r1c['e2e_acc'] - base_acc, 4),
                 'n_lost': n_lost_base, 'n_rejected': r1c['n_rejected'],
                 'rejection_precision': round(r1c['rej_precision'], 4),
                 'notes': f"conf_th={conf_th}, method=product"})

    # Combo best
    combo_s1 = 0.32 if sc == 'CBC_Only' else 0.50
    combo_conf = 0.33 if sc == 'CBC_Only' else 0.27
    rc = evaluate_combined(base_df, combo_s1, combo_conf, 'product')
    if rc:
        rows.append({'scenario': sc, 'strategy': '1a+1c Combined (best)',
                     'test_acc': round(rc['e2e_acc'], 4), 'test_f1': round(rc['e2e_f1'], 4),
                     'delta_acc': round(rc['e2e_acc'] - base_acc, 4),
                     'n_lost': rc['n_lost'], 'n_rejected': rc['n_rejected'],
                     'rejection_precision': np.nan,
                     'notes': f"S1={combo_s1}, conf={combo_conf}"})

strategy_df = pd.DataFrame(rows)
strategy_df.to_excel(analysis_dir / 'e2e_strategy1_comparison.xlsx', index=False)
print("✅ e2e_strategy1_comparison.xlsx")

# ── 2. Summary Figure — grouped bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

strategies = ['Baseline (M4)', '1a E2E-optimal threshold', '1b OAC rejection',
              '1c E2E confidence (product)', '1a+1c Combined (best)']
short_labels = ['Baseline', '1a\nE2E-opt th', '1b\nOAC reject', '1c\nE2E conf', '1a+1c\nCombined']

bar_colors = [PALETTE['base2'], PALETTE['base1'], PALETTE['accent1'],
              PALETTE['accent2'], PALETTE['highlight']]

for idx, sc in enumerate(SCENARIOS):
    sc_data = strategy_df[strategy_df['scenario'] == sc]

    accs = []
    for s in strategies:
        row = sc_data[sc_data['strategy'] == s]
        accs.append(row['test_acc'].values[0] if len(row) > 0 else 0)

    x = np.arange(len(strategies))
    bars = axes[idx].bar(x, accs, color=bar_colors, edgecolor='white', width=0.6)

    # Baseline reference line
    axes[idx].axhline(accs[0], color='black', linestyle=':', linewidth=0.8, alpha=0.5)

    # Value labels
    for bar, acc in zip(bars, accs):
        delta = acc - accs[0]
        delta_str = f"+{delta:.3f}" if delta > 0 else f"{delta:.3f}" if delta < 0 else ""
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                      f"{acc:.3f}\n{delta_str}", ha='center', va='bottom', fontsize=8,
                      fontweight='bold' if abs(delta) > 0.005 else 'normal')

    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(short_labels, fontsize=8)
    axes[idx].set_ylabel('E2E Accuracy (Test)' if idx == 0 else '', fontsize=9)
    axes[idx].set_title(f'{sc}', fontsize=11, fontweight='bold')
    axes[idx].set_ylim(min(accs) - 0.03, max(accs) + 0.03)
    despine_all(axes[idx])

fig.suptitle('Strategy 1 Comparison — Pipeline Optimization Results (Test)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'strategy1_summary_comparison')
plt.show()
print("✅ Figure saved")

# ── Print summary ──
print("\n" + "=" * 65)
print("Strategy 1 — Final Summary")
print("=" * 65)
print(strategy_df[['scenario', 'strategy', 'test_acc', 'delta_acc', 'n_lost', 'notes']].to_string(index=False))

## CELL S4a — MEDIUM Zone → Biochemistry Upgrade Simulation

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S4a — MEDIUM Zone → Biochemistry Upgrade Simulation
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Strategy 4a — MEDIUM Zone Patients → Re-evaluate with CBC_BIO")
print("=" * 65)

# Logic: for patients falling in MEDIUM zone in the CBC_Only pipeline,
# use the CBC_BIO prediction (simulate as if biochemistry was requested)

# First align CBC_Only and CBC_BIO test e2e per patient
# (same 229 patients, same order)

cbc_only_test = e2e_results['CBC_Only_test'].copy()
cbc_bio_test  = e2e_results['CBC_BIO_test'].copy()

# Verify: same patients?
assert len(cbc_only_test) == len(cbc_bio_test) == 229
assert (cbc_only_test['e2e_true'].values == cbc_bio_test['e2e_true'].values).all(), \
    "Patient order mismatch!"

print(f"\nCBC_Only Baseline:")
print(f"  E2E Acc: {cbc_only_test['e2e_correct'].mean():.3f}")
print(f"  Zone distribution:")
for z in ['Excluded', 'HIGH', 'MEDIUM', 'LOW']:
    n = (cbc_only_test['e2e_zone'] == z).sum()
    acc = cbc_only_test[cbc_only_test['e2e_zone'] == z]['e2e_correct'].mean() if n > 0 else 0
    print(f"    {z:10s}: n={n:3d}  acc={acc:.3f}")

# ── 4a Simulation ──
# Take MEDIUM zone patients' e2e_pred from CBC_BIO
hybrid = cbc_only_test.copy()
medium_mask = (hybrid['e2e_zone'] == 'MEDIUM')
n_medium = medium_mask.sum()

# CBC_BIO prediction for these patients
hybrid.loc[medium_mask, 'e2e_pred_4a'] = cbc_bio_test.loc[medium_mask, 'e2e_pred'].values
hybrid.loc[~medium_mask, 'e2e_pred_4a'] = hybrid.loc[~medium_mask, 'e2e_pred']

hybrid['e2e_correct_4a'] = (hybrid['e2e_true'] == hybrid['e2e_pred_4a'])

acc_4a = hybrid['e2e_correct_4a'].mean()
f1_4a = f1_score(hybrid['e2e_true'], hybrid['e2e_pred_4a'], labels=E2E_CLASSES, average='macro')

# Improvement in MEDIUM zone
medium_acc_before = cbc_only_test.loc[medium_mask, 'e2e_correct'].mean()
medium_acc_after  = hybrid.loc[medium_mask, 'e2e_correct_4a'].mean()

print(f"\n4a — MEDIUM → CBC_BIO upgrade ({n_medium} patients):")
print(f"  MEDIUM zone acc: {medium_acc_before:.3f} → {medium_acc_after:.3f} (+{medium_acc_after - medium_acc_before:.3f})")
print(f"  Overall E2E Acc: {cbc_only_test['e2e_correct'].mean():.3f} → {acc_4a:.3f} (+{acc_4a - cbc_only_test['e2e_correct'].mean():.3f})")
print(f"  Overall E2E F1:  {f1_score(cbc_only_test['e2e_true'], cbc_only_test['e2e_pred'], labels=E2E_CLASSES, average='macro'):.3f} → {f1_4a:.3f}")

# ── Verify the same on OOF ──
cbc_only_oof = e2e_results['CBC_Only_oof'].copy()
cbc_bio_oof  = e2e_results['CBC_BIO_oof'].copy()

hybrid_oof = cbc_only_oof.copy()
med_mask_oof = (hybrid_oof['e2e_zone'] == 'MEDIUM')
hybrid_oof.loc[med_mask_oof, 'e2e_pred_4a'] = cbc_bio_oof.loc[med_mask_oof, 'e2e_pred'].values
hybrid_oof.loc[~med_mask_oof, 'e2e_pred_4a'] = hybrid_oof.loc[~med_mask_oof, 'e2e_pred']
hybrid_oof['e2e_correct_4a'] = (hybrid_oof['e2e_true'] == hybrid_oof['e2e_pred_4a'])

acc_4a_oof = hybrid_oof['e2e_correct_4a'].mean()
print(f"\n  OOF verification: {cbc_only_oof['e2e_correct'].mean():.3f} → {acc_4a_oof:.3f} (+{acc_4a_oof - cbc_only_oof['e2e_correct'].mean():.3f})")

# ── Zone expansion: include MEDIUM + Excluded too ──
print(f"\n{'─'*55}")
print("4a Variations — Which zones to upgrade?")
print(f"{'─'*55}")

zone_combos = [
    ('MEDIUM only', ['MEDIUM']),
    ('MEDIUM + LOW', ['MEDIUM', 'LOW']),
    ('MEDIUM + Excluded', ['MEDIUM', 'Excluded']),
    ('All non-HIGH', ['MEDIUM', 'LOW', 'Excluded']),
]

for label, zones in zone_combos:
    h = cbc_only_test.copy()
    upgrade_mask = h['e2e_zone'].isin(zones)
    n_upgrade = upgrade_mask.sum()

    h.loc[upgrade_mask, 'e2e_pred_v'] = cbc_bio_test.loc[upgrade_mask, 'e2e_pred'].values
    h.loc[~upgrade_mask, 'e2e_pred_v'] = h.loc[~upgrade_mask, 'e2e_pred']

    acc_v = accuracy_score(h['e2e_true'], h['e2e_pred_v'])
    f1_v = f1_score(h['e2e_true'], h['e2e_pred_v'], labels=E2E_CLASSES, average='macro')

    # Biochemistry request rate
    bio_rate = n_upgrade / len(h) * 100

    print(f"  {label:22s}: n={n_upgrade:3d} ({bio_rate:4.1f}%)  "
          f"acc={acc_v:.3f} (+{acc_v - cbc_only_test['e2e_correct'].mean():.3f})  f1={f1_v:.3f}")

## CELL S4b — S1 Borderline OAC → Safety Net + Save All S4

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL S4b — Fix & Rerun
# ══════════════════════════════════════════════════════════

df = e2e_results['CBC_Only_test'].copy()
bio = e2e_results['CBC_BIO_test']
baseline_acc = df['e2e_correct'].mean()
baseline_f1 = f1_score(df['e2e_true'], df['e2e_pred'], labels=E2E_CLASSES, average='macro')

op = get_operating_point(th_config, 'CBC_Only')
s1_th = op['s1_threshold']

def to_array(x):
    """Ensure numpy array."""
    return x.values if hasattr(x, 'values') else np.array(x)

combos = [
    ("Baseline", df['e2e_pred'].values),
    ("4a: MEDIUM→BIO",
     np.where(df['e2e_zone'] == 'MEDIUM', bio['e2e_pred'].values, df['e2e_pred'].values)),
    ("4a+: MEDIUM+Excluded→BIO",
     np.where(df['e2e_zone'].isin(['MEDIUM', 'Excluded']), bio['e2e_pred'].values, df['e2e_pred'].values)),
    ("4a+4b: All non-HIGH + safety→BIO",
     np.where(
         (df['e2e_zone'].isin(['MEDIUM', 'LOW', 'Excluded'])) |
         ((df['s1_pred'] == 'DAS') & (df['s1_prob'] >= s1_th - 0.15)),
         bio['e2e_pred'].values, df['e2e_pred'].values)),
    ("Full CBC_BIO (reference)", bio['e2e_pred'].values),
]

print(f"{'Strategy':<35s} {'Acc':>6s} {'Δ':>7s} {'F1':>6s} {'BIO needed':>12s}")
print("─" * 70)

s4_rows = []
for label, pred in combos:
    acc = accuracy_score(df['e2e_true'], pred)
    f1 = f1_score(df['e2e_true'], pred, labels=E2E_CLASSES, average='macro')
    n_bio = (pred != df['e2e_pred'].values).sum() if label != 'Baseline' else 0
    bio_pct = n_bio / len(df) * 100
    delta = acc - baseline_acc
    print(f"  {label:<33s} {acc:>6.3f} {delta:>+7.3f} {f1:>6.3f} {n_bio:>4d} ({bio_pct:4.1f}%)")
    s4_rows.append({'strategy': label, 'test_acc': round(acc, 4), 'test_f1': round(f1, 4),
                    'delta_acc': round(delta, 4), 'n_bio_needed': n_bio,
                    'bio_pct': round(bio_pct, 1)})

# ── Excel ──
s4_df = pd.DataFrame(s4_rows)
s4_df.to_excel(analysis_dir / 'e2e_strategy4_comparison.xlsx', index=False)
print("\n✅ e2e_strategy4_comparison.xlsx")

# ── Figure ──
fig, ax = plt.subplots(figsize=(10, 5))

labels_short = ['Baseline\nCBC_Only', '4a\nMEDIUM→BIO', '4a+\nMED+Excl→BIO',
                '4a+4b\nAll+Safety→BIO', 'Full\nCBC_BIO']
accs = s4_df['test_acc'].values
bio_pcts = s4_df['bio_pct'].values

colors = [PALETTE['base2'], PALETTE['accent2'], PALETTE['accent1'],
          PALETTE['highlight'], PALETTE['accent3']]

bars = ax.bar(range(len(accs)), accs, color=colors, edgecolor='white', width=0.6)

ax.axhline(accs[0], color=PALETTE['base2'], linestyle=':', linewidth=1, alpha=0.7)
ax.axhline(accs[-1], color=PALETTE['accent3'], linestyle=':', linewidth=1, alpha=0.7)

for i, (bar, acc, bio_pct) in enumerate(zip(bars, accs, bio_pcts)):
    delta = acc - accs[0]
    delta_str = f"+{delta:.3f}" if delta > 0 else ""
    bio_str = f"BIO: {bio_pct:.0f}%" if 0 < i < len(accs)-1 else ""
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.3f}\n{delta_str}\n{bio_str}", ha='center', va='bottom', fontsize=9,
            fontweight='bold' if delta > 0.05 else 'normal')

ax.set_xticks(range(len(labels_short)))
ax.set_xticklabels(labels_short, fontsize=9)
ax.set_ylabel('E2E Accuracy (Test)', fontsize=10)
ax.set_ylim(0.50, 0.76)
ax.set_title('Strategy 4 — Clinical Workflow Optimization\nCBC_Only + Selective Biochemistry Upgrade',
             fontsize=12, fontweight='bold')
despine_all(ax)

fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'strategy4_clinical_workflow')
plt.show()
print("✅ Figure saved")

#TIER 1 2

## CELL T1 — Tier 1 & Tier 2 Zone Assignments

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T1 — Tier 1 & Tier 2 Zone Assignments
# ══════════════════════════════════════════════════════════
import os

# Directories
zone_dir = PATHS.module_dir('m4_e2e', 'zone_assignments')   # tier parquets persist here
os.makedirs(zone_dir, exist_ok=True)

E2E_ZONES_ORDER = ['Excluded', 'HIGH', 'MEDIUM', 'LOW']

# ── Tier 1 (CBC_Only) zone assignments ──
for sp in ['oof', 'test']:
    tier1 = e2e_results[f'CBC_Only_{sp}'].copy()
    tier1 = tier1[['idx_s1', 's1_true', 'diagnosis', 's1_prob', 's1_pred',
                    's2_true', 's2_pred', 's2_confidence', 'e2e_true', 'e2e_pred',
                    'e2e_zone', 'e2e_correct', 'is_lost']].copy()
    tier1.rename(columns={'e2e_zone': 'tier1_zone', 'e2e_pred': 'tier1_pred',
                          'e2e_correct': 'tier1_correct'}, inplace=True)
    tier1['tier'] = 'Tier1_CBC_Only'

    out_path = zone_dir / f'tier1_zone_assignments_{sp}.parquet'
    tier1.to_parquet(out_path, index=False)
    print(f"✅ {out_path.name}: {len(tier1)} rows")

# ── Tier 2 (CBC_BIO) zone assignments ──
for sp in ['oof', 'test']:
    tier2 = e2e_results[f'CBC_BIO_{sp}'].copy()
    tier2 = tier2[['idx_s1', 's1_true', 'diagnosis', 's1_prob', 's1_pred',
                    's2_true', 's2_pred', 's2_confidence', 'e2e_true', 'e2e_pred',
                    'e2e_zone', 'e2e_correct', 'is_lost']].copy()
    tier2.rename(columns={'e2e_zone': 'tier2_zone', 'e2e_pred': 'tier2_pred',
                          'e2e_correct': 'tier2_correct'}, inplace=True)
    tier2['tier'] = 'Tier2_CBC_BIO'

    out_path = zone_dir / f'tier2_zone_assignments_{sp}.parquet'
    tier2.to_parquet(out_path, index=False)
    print(f"✅ {out_path.name}: {len(tier2)} rows")

# ── Verification: consistency with M4 E2E reference values ──
print("\n" + "=" * 65)
print("M4 E2E Reference Verification (Test)")
print("=" * 65)

for tier_label, sc in [('Tier1 CBC_Only', 'CBC_Only'), ('Tier2 CBC_BIO', 'CBC_BIO')]:
    df = e2e_results[f'{sc}_test']
    zone_col = 'e2e_zone'
    correct_col = 'e2e_correct'

    print(f"\n{tier_label}:")
    for z in E2E_ZONES_ORDER:
        zdf = df[df[zone_col] == z]
        n = len(zdf)
        pct = n / len(df) * 100
        acc = zdf[correct_col].mean() if n > 0 else np.nan
        print(f"  {z:10s}: n={n:3d} ({pct:5.1f}%)  acc={acc:.3f}" if n > 0
              else f"  {z:10s}: n={n:3d} ({pct:5.1f}%)")

## CELL T2 — Tier Transition Matrix (Tier 1 Zone → Tier 2 Zone)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T2 — Tier Transition Matrix (Tier1 Zone → Tier2 Zone)
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Tier Transition Matrix — Tier1 Zone → Tier2 Zone")
print("=" * 65)

transition_dfs = {}

for sp in ['oof', 'test']:
    t1 = e2e_results[f'CBC_Only_{sp}'].copy()
    t2 = e2e_results[f'CBC_BIO_{sp}'].copy()

    # Same patients (positional alignment)
    merged = pd.DataFrame({
        'tier1_zone': t1['e2e_zone'].values,
        'tier2_zone': t2['e2e_zone'].values,
        'tier1_pred': t1['e2e_pred'].values,
        'tier2_pred': t2['e2e_pred'].values,
        'tier1_correct': t1['e2e_correct'].values,
        'tier2_correct': t2['e2e_correct'].values,
        'e2e_true': t1['e2e_true'].values,
    })

    # Cross-tabulation
    ct = pd.crosstab(merged['tier1_zone'], merged['tier2_zone'],
                      margins=True, margins_name='Total')
    # Sort
    idx_order = [z for z in E2E_ZONES_ORDER if z in ct.index] + ['Total']
    col_order = [z for z in E2E_ZONES_ORDER if z in ct.columns] + ['Total']
    ct = ct.reindex(index=idx_order, columns=col_order, fill_value=0)

    transition_dfs[sp] = (ct, merged)

    print(f"\n{sp.upper()} (n={len(merged)}):")
    print(ct.to_string())

    # MEDIUM→HIGH transition rate (critical metric)
    medium_mask = (merged['tier1_zone'] == 'MEDIUM')
    n_medium = medium_mask.sum()
    if n_medium > 0:
        medium_to_high = (merged.loc[medium_mask, 'tier2_zone'] == 'HIGH').sum()
        medium_acc_t1 = merged.loc[medium_mask, 'tier1_correct'].mean()
        medium_acc_t2 = merged.loc[medium_mask, 'tier2_correct'].mean()
        print(f"\n  MEDIUM patients (n={n_medium}):")
        print(f"    → HIGH (Tier2): {medium_to_high} ({medium_to_high/n_medium*100:.1f}%)")
        print(f"    Acc Tier1: {medium_acc_t1:.3f} → Acc Tier2: {medium_acc_t2:.3f} "
              f"(+{medium_acc_t2 - medium_acc_t1:.3f})")

# Save Excel
metrics_dir = PATHS.module_dir('m4_e2e', 'analysis')
with pd.ExcelWriter(metrics_dir / 'tier_transition_matrix.xlsx') as w:
    for sp in ['oof', 'test']:
        ct, _ = transition_dfs[sp]
        ct.to_excel(w, sheet_name=f'transition_{sp}')
print("\n✅ tier_transition_matrix.xlsx")

## CELL T3 — Figure: Tier 1 vs Tier 2 Zone Distribution (Stacked Bar)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T3 — Figure: Tier 1 vs Tier 2 Zone Distribution (Stacked Bar)
# ══════════════════════════════════════════════════════════

zone_colors = {'Excluded': PALETTE['base2'], 'HIGH': PALETTE['accent1'],
               'MEDIUM': PALETTE['accent2'], 'LOW': PALETTE['highlight']}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for sp_idx, sp in enumerate(['test']):  # test only
    ax = axes[0]  # Tier 1
    ax2 = axes[1] # Tier 2

    for ax_i, (sc, tier_label) in enumerate([(f'CBC_Only_{sp}', 'Tier 1 (CBC_Only)'),
                                               (f'CBC_BIO_{sp}', 'Tier 2 (CBC_BIO)')]):
        cur_ax = axes[ax_i]
        df = e2e_results[sc]
        zone_col = 'e2e_zone'

        bottom = 0
        for z in E2E_ZONES_ORDER:
            zdf = df[df[zone_col] == z]
            n = len(zdf)
            pct = n / len(df) * 100
            acc = zdf['e2e_correct'].mean() if n > 0 else 0

            bar = cur_ax.bar(0, pct, bottom=bottom, color=zone_colors[z],
                            edgecolor='white', width=0.5)

            if pct > 4:
                cur_ax.text(0, bottom + pct/2,
                           f"{z}\nn={n} ({pct:.1f}%)\nacc={acc:.2f}",
                           ha='center', va='center', fontsize=7.5, fontweight='bold')
            bottom += pct

        cur_ax.set_ylim(0, 105)
        cur_ax.set_xlim(-0.5, 0.5)
        cur_ax.set_xticks([])
        cur_ax.set_ylabel('Patient proportion (%)' if ax_i == 0 else '', fontsize=9)
        cur_ax.set_title(f'{tier_label}\nE2E Acc={df["e2e_correct"].mean():.3f}',
                        fontsize=10, fontweight='bold')
        despine_all(cur_ax)

fig.suptitle('Tier 1 vs Tier 2 — Zone Distribution (Test, n=229)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'tier_zone_comparison')
plt.show()
print("✅ Figure saved")

## CELL T4 — Figure: Tier Transition Heatmap


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T4 — Figure: Tier Transition Heatmap
# ══════════════════════════════════════════════════════════

from matplotlib.colors import LinearSegmentedColormap

_, merged_test = transition_dfs['test']

ct_raw = pd.crosstab(merged_test['tier1_zone'], merged_test['tier2_zone'])
zones_present = [z for z in E2E_ZONES_ORDER if z in ct_raw.index or z in ct_raw.columns]
ct_raw = ct_raw.reindex(index=zones_present, columns=zones_present, fill_value=0)

# Row-normalize
ct_norm = ct_raw.div(ct_raw.sum(axis=1), axis=0).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Raw counts
im1 = axes[0].imshow(ct_raw.values, cmap=cmap_tufte, aspect='equal')
for i in range(len(zones_present)):
    for j in range(len(zones_present)):
        val = ct_raw.values[i, j]
        color = 'white' if val > ct_raw.values.max() * 0.6 else 'black'
        axes[0].text(j, i, str(val), ha='center', va='center', fontsize=10,
                    color=color, fontweight='bold')

axes[0].set_xticks(range(len(zones_present)))
axes[0].set_xticklabels(zones_present, rotation=45, ha='right', fontsize=9)
axes[0].set_yticks(range(len(zones_present)))
axes[0].set_yticklabels(zones_present, fontsize=9)
axes[0].set_xlabel('Tier 2 (CBC_BIO) Zone', fontsize=9)
axes[0].set_ylabel('Tier 1 (CBC_Only) Zone', fontsize=9)
axes[0].set_title('Raw Counts', fontsize=10, fontweight='bold')
despine_all(axes[0])

# Normalized (row-wise)
im2 = axes[1].imshow(ct_norm.values, cmap=cmap_tufte, vmin=0, vmax=1, aspect='equal')
for i in range(len(zones_present)):
    for j in range(len(zones_present)):
        raw = ct_raw.values[i, j]
        norm = ct_norm.values[i, j]
        color = 'white' if norm > 0.5 else 'black'
        axes[1].text(j, i, f"{raw}\n({norm:.2f})", ha='center', va='center',
                    fontsize=8, color=color, fontweight='bold')

axes[1].set_xticks(range(len(zones_present)))
axes[1].set_xticklabels(zones_present, rotation=45, ha='right', fontsize=9)
axes[1].set_yticks(range(len(zones_present)))
axes[1].set_yticklabels(zones_present, fontsize=9)
axes[1].set_xlabel('Tier 2 (CBC_BIO) Zone', fontsize=9)
axes[1].set_ylabel('Tier 1 (CBC_Only) Zone', fontsize=9)
axes[1].set_title('Row-Normalized', fontsize=10, fontweight='bold')
despine_all(axes[1])

fig.colorbar(im2, ax=axes[1], shrink=0.8, pad=0.02).set_label('Proportion', fontsize=8)

fig.suptitle('Tier Transition Matrix — Tier 1 Zone → Tier 2 Zone (Test)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'tier_transition_heatmap')
plt.show()
print("✅ Figure saved")

## CELL T5 — Figure: Per-Zone Accuracy — Tier 1 vs Tier 2


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T5 — Figure: Zone-Based Accuracy — Tier 1 vs Tier 2
# ══════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 5))

zones_plot = ['Excluded', 'HIGH', 'MEDIUM']  # LOW practically empty
x = np.arange(len(zones_plot))
width = 0.35

tier1_accs = []
tier2_accs = []
tier1_ns = []
tier2_ns = []

t1_test = e2e_results['CBC_Only_test']
t2_test = e2e_results['CBC_BIO_test']

for z in zones_plot:
    z1 = t1_test[t1_test['e2e_zone'] == z]
    z2 = t2_test[t2_test['e2e_zone'] == z]
    tier1_accs.append(z1['e2e_correct'].mean() if len(z1) > 0 else 0)
    tier2_accs.append(z2['e2e_correct'].mean() if len(z2) > 0 else 0)
    tier1_ns.append(len(z1))
    tier2_ns.append(len(z2))

bars1 = ax.bar(x - width/2, tier1_accs, width, color=PALETTE['base1'],
               edgecolor='white', label='Tier 1 (CBC_Only)')
bars2 = ax.bar(x + width/2, tier2_accs, width, color=PALETTE['accent3'],
               edgecolor='white', label='Tier 2 (CBC_BIO)')

# Labels
for i, (b1, b2, n1, n2) in enumerate(zip(bars1, bars2, tier1_ns, tier2_ns)):
    ax.text(b1.get_x() + b1.get_width()/2, b1.get_height() + 0.01,
            f"{tier1_accs[i]:.3f}\nn={n1}", ha='center', va='bottom', fontsize=8)
    ax.text(b2.get_x() + b2.get_width()/2, b2.get_height() + 0.01,
            f"{tier2_accs[i]:.3f}\nn={n2}", ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(zones_plot, fontsize=10)
ax.set_ylabel('E2E Accuracy', fontsize=10)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9, frameon=False)
ax.set_title('Zone-Based Accuracy — Tier 1 vs Tier 2 (Test)',
             fontsize=12, fontweight='bold')
despine_all(ax)

fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'tier_zone_accuracy_comparison')
plt.show()
print("✅ Figure saved")

## CELL T6 — Tier Summary & Manifest

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL T6 — Tier Summary & Manifest
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Tier Analysis — Summary")
print("=" * 65)

_, merged_test = transition_dfs['test']

# MEDIUM→HIGH transition (main metric)
med = merged_test[merged_test['tier1_zone'] == 'MEDIUM']
med_to_high = (med['tier2_zone'] == 'HIGH').sum()
med_acc_t1 = med['tier1_correct'].mean()
med_acc_t2 = med['tier2_correct'].mean()

# Excluded patients
excl = merged_test[merged_test['tier1_zone'] == 'Excluded']
excl_to_high = (excl['tier2_zone'] == 'HIGH').sum() if len(excl) > 0 else 0

print(f"""
Tier 1 MEDIUM → Tier 2 transition ({len(med)} patients):
  → HIGH:     {med_to_high} ({med_to_high/len(med)*100:.1f}%)
  → MEDIUM:   {(med['tier2_zone'] == 'MEDIUM').sum()} ({(med['tier2_zone'] == 'MEDIUM').sum()/len(med)*100:.1f}%)
  → Excluded: {(med['tier2_zone'] == 'Excluded').sum()}
  Acc: {med_acc_t1:.3f} → {med_acc_t2:.3f} (+{med_acc_t2 - med_acc_t1:.3f})

Tier 1 Excluded → Tier 2 transition ({len(excl)} patients):
  → HIGH:     {excl_to_high}
  → Excluded: {(excl['tier2_zone'] == 'Excluded').sum()}

Outputs:
  zone_assignments/tier1_zone_assignments_oof.parquet
  zone_assignments/tier1_zone_assignments_test.parquet
  zone_assignments/tier2_zone_assignments_oof.parquet
  zone_assignments/tier2_zone_assignments_test.parquet
  analysis/tier_transition_matrix.xlsx
  plots/tier_zone_comparison
  plots/tier_transition_heatmap
  plots/tier_zone_accuracy_comparison
""")

# Chat 4 Extension — Cascade Simulation + Efficiency Curve


##  CELL C1 — Cascade Simulation (Patient-Level)


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL C1 — Cascade Simulation (Patient-Based)
# ══════════════════════════════════════════════════════════
import os

reflex_dir = PATHS.module_dir('m4_e2e', 'reflex')
os.makedirs(reflex_dir, exist_ok=True)

def simulate_cascade(t1_df, t2_df, escalation_zones=['MEDIUM', 'LOW']):
    """
    Tier 1 → escalation → Tier 2 cascade simulation.
    escalation_zones: patients falling in these Tier 1 zones are sent to Tier 2.
    """
    cascade = pd.DataFrame({
        'patient_idx':  np.arange(len(t1_df)),
        'true_label':   t1_df['e2e_true'].values,
        'diagnosis':    t1_df['diagnosis'].values,
        'tier1_pred':   t1_df['e2e_pred'].values,
        'tier1_zone':   t1_df['e2e_zone'].values,
        'tier1_correct': t1_df['e2e_correct'].values,
        'tier1_s1_prob': t1_df['s1_prob'].values,
        'tier1_s2_conf': t1_df['s2_confidence'].values,
        'tier2_pred':   t2_df['e2e_pred'].values,
        'tier2_zone':   t2_df['e2e_zone'].values,
        'tier2_correct': t2_df['e2e_correct'].values,
    })

    # Escalation decision
    cascade['escalated'] = cascade['tier1_zone'].isin(escalation_zones)

    # Final prediction: if escalated use Tier 2, otherwise Tier 1
    cascade['final_pred'] = np.where(cascade['escalated'],
                                      cascade['tier2_pred'], cascade['tier1_pred'])
    cascade['final_zone'] = np.where(cascade['escalated'],
                                      cascade['tier2_zone'], cascade['tier1_zone'])
    cascade['final_correct'] = (cascade['true_label'] == cascade['final_pred'])

    return cascade

# ── Cascade simulation for test and OOF ──
for sp in ['oof', 'test']:
    t1 = e2e_results[f'CBC_Only_{sp}']
    t2 = e2e_results[f'CBC_BIO_{sp}']

    cascade = simulate_cascade(t1, t2, escalation_zones=['MEDIUM', 'LOW', 'Excluded'])

    # Save
    cascade.to_parquet(reflex_dir / f'cascade_simulation_{sp}.parquet', index=False)

    n_esc = cascade['escalated'].sum()
    bio_pct = n_esc / len(cascade) * 100
    acc = cascade['final_correct'].mean()

    print(f"{'─'*55}")
    print(f"{sp.upper()} Cascade (n={len(cascade)}):")
    print(f"  Escalated: {n_esc} ({bio_pct:.1f}%)")
    print(f"  Final Acc: {acc:.3f}")
    print(f"  Zone distribution (final):")
    for z in ['Excluded', 'HIGH', 'MEDIUM', 'LOW']:
        zdf = cascade[cascade['final_zone'] == z]
        if len(zdf) > 0:
            print(f"    {z:10s}: n={len(zdf):3d}  acc={zdf['final_correct'].mean():.3f}")

print(f"\n✅ cascade_simulation_oof.parquet")
print(f"✅ cascade_simulation_test.parquet")

## CELL C2 — Efficiency Curve (MAIN PUBLICATION FIGURE)


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL C2 — Efficiency Curve (MAIN PUBLICATION FIGURE)
# ══════════════════════════════════════════════════════════

def efficiency_curve_data(t1_df, t2_df):
    """
    Build an efficiency curve by sweeping the S2 confidence threshold.
    Low threshold = less escalation, high = more escalation.
    """
    records = []
    n = len(t1_df)

    # Reference: 0% biochemistry (pure CBC_Only)
    records.append({'bio_pct': 0, 'n_bio': 0,
                    'acc': t1_df['e2e_correct'].mean(),
                    'label': 'Pure CBC_Only'})

    # Strategy: zone-based escalation combinations
    zone_combos = [
        ('LOW only', ['LOW']),
        ('LOW+MEDIUM', ['LOW', 'MEDIUM']),
        ('LOW+MEDIUM+Excluded', ['LOW', 'MEDIUM', 'Excluded']),
    ]

    for label, zones in zone_combos:
        cascade = simulate_cascade(t1_df, t2_df, escalation_zones=zones)
        n_bio = cascade['escalated'].sum()
        records.append({
            'bio_pct': n_bio / n * 100,
            'n_bio': n_bio,
            'acc': cascade['final_correct'].mean(),
            'label': label
        })

    # Continuous curve: sweep Tier 1 confidence threshold
    # Low confidence → escalation (sorted by S2 confidence)
    t1_zones = t1_df['e2e_zone'].values
    t1_correct = t1_df['e2e_correct'].values
    t2_correct = t2_df['e2e_correct'].values
    t1_s2_conf = t1_df['s2_confidence'].astype(float).fillna(0).values
    t1_s1_prob = t1_df['s1_prob'].values

    # Compound score: s1_prob * s2_confidence (use s1_prob for Excluded patients)
    compound = np.where(t1_zones == 'Excluded',
                        1 - t1_s1_prob,  # DAS confidence
                        t1_s1_prob * t1_s2_conf)

    # Add escalation in increasing compound score order
    sorted_idx = np.argsort(compound)

    current_correct = t1_correct.copy().astype(float)

    for k in range(1, n + 1):
        # Escalate the k patients with the lowest compound score
        esc_indices = sorted_idx[:k]
        final_correct = t1_correct.copy().astype(float)
        final_correct[esc_indices] = t2_correct[esc_indices]

        if k % 5 == 0 or k == n:  # record every 5 patients
            records.append({
                'bio_pct': k / n * 100,
                'n_bio': k,
                'acc': final_correct.mean(),
                'label': 'continuous'
            })

    return pd.DataFrame(records)

# ── Compute curves ──
eff_test = efficiency_curve_data(e2e_results['CBC_Only_test'], e2e_results['CBC_BIO_test'])
eff_oof = efficiency_curve_data(e2e_results['CBC_Only_oof'], e2e_results['CBC_BIO_oof'])

# Save
eff_test.to_parquet(reflex_dir / 'efficiency_curve_data_test.parquet', index=False)
eff_oof.to_parquet(reflex_dir / 'efficiency_curve_data_oof.parquet', index=False)

# ── PUBLICATION FIGURE ──
fig, ax = plt.subplots(figsize=(9, 5.5))

# Continuous curve
cont = eff_test[eff_test['label'] == 'continuous']
ax.plot(cont['bio_pct'], cont['acc'], color=PALETTE['base1'], linewidth=2, zorder=2)

# OOF continuous curve (reference)
cont_oof = eff_oof[eff_oof['label'] == 'continuous']
ax.plot(cont_oof['bio_pct'], cont_oof['acc'], color=PALETTE['base1'],
        linewidth=1, linestyle='--', alpha=0.5, label='OOF', zorder=1)

# Reference points
ref_points = eff_test[eff_test['label'] != 'continuous'].copy()
ref_colors = {
    'Pure CBC_Only': PALETTE['base2'],
    'LOW only': PALETTE['highlight'],
    'LOW+MEDIUM': PALETTE['accent2'],
    'LOW+MEDIUM+Excluded': PALETTE['accent1'],
}

for _, row in ref_points.iterrows():
    color = ref_colors.get(row['label'], PALETTE['accent3'])
    ax.scatter(row['bio_pct'], row['acc'], color=color, s=120, zorder=5,
              edgecolors='black', linewidth=1.2)

# Annotations for the three main reference points
# 1. Pure CBC_Only (0%)
pure = ref_points[ref_points['label'] == 'Pure CBC_Only'].iloc[0]
ax.annotate(f"Pure CBC_Only\nacc={pure['acc']:.3f}",
            xy=(pure['bio_pct'], pure['acc']),
            xytext=(8, pure['acc'] - 0.03),
            fontsize=8, fontweight='bold', color=PALETTE['base2'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['base2'], lw=1.2))

# 2. Sweet spot (~29%)
sweet = ref_points[ref_points['label'] == 'LOW+MEDIUM+Excluded'].iloc[0]
ax.annotate(f"Sweet Spot\nacc={sweet['acc']:.3f}\n{sweet['bio_pct']:.0f}% biochemistry\n→ 94% of the gain",
            xy=(sweet['bio_pct'], sweet['acc']),
            xytext=(sweet['bio_pct'] - 32, sweet['acc'] + 0.005),
            fontsize=8, fontweight='bold', color=PALETTE['accent1'],
            ha='left', va='center',
            arrowprops=dict(arrowstyle='->', color=PALETTE['accent1'], lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PALETTE['accent1'], alpha=0.9))

# 3. Full CBC_BIO (100%) — last point
full_bio_acc = e2e_results['CBC_BIO_test']['e2e_correct'].mean()
ax.scatter(100, full_bio_acc, color=PALETTE['accent3'], s=120, zorder=5,
          edgecolors='black', linewidth=1.2)
ax.annotate(f"Full CBC_BIO\nacc={full_bio_acc:.3f}",
            xy=(100, full_bio_acc),
            xytext=(99, full_bio_acc - 0.05),
            fontsize=8, fontweight='bold', color=PALETTE['accent3'],
            ha='right',
            arrowprops=dict(arrowstyle='->', color=PALETTE['accent3'], lw=1.2))

# Axes
ax.set_xlabel('Proportion of Patients Referred for Biochemistry (%)', fontsize=11)
ax.set_ylabel('E2E Accuracy', fontsize=11)
ax.set_title('Reflex Test Efficiency Curve — Selective Biochemistry Strategy',
             fontsize=13, fontweight='bold')
ax.set_xlim(-3, 105)
ax.set_ylim(0.53, 0.73)
despine_all(ax)

fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'reflex_efficiency_curve')
plt.show()
print("✅ Efficiency curve (publication figure) saved")

## CELL C3 — Lost Patient Analysis (Cascade Perspective)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL C3 — Lost Patient Analysis (Cascade Perspective)
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("Lost Patient Analysis — Cascade Perspective (Test)")
print("=" * 65)

cascade_test = pd.read_parquet(reflex_dir / 'cascade_simulation_test.parquet')
t1_test = e2e_results['CBC_Only_test']
t2_test = e2e_results['CBC_BIO_test']

# Internal class key → display label
CLS_DISP = {'DAS': 'OAC', 'DEA': 'IDA', 'HA': 'HA', 'HGB HTZ': 'HGB HTZ', 'Normal': 'Normal'}

# 1. True IAS patients excluded at Tier 1
t1_excluded_ias = t1_test[(t1_test['e2e_zone'] == 'Excluded') & (t1_test['s1_true'] == 'IAS')]
print(f"\n1. IAS patients excluded at Tier 1: {len(t1_excluded_ias)}")
if len(t1_excluded_ias) > 0:
    _d = {CLS_DISP.get(k, k): v for k, v in t1_excluded_ias['s2_true'].value_counts().to_dict().items()}
    print(f"   True S2 classes: {_d}")

# 2. Patients rescued after cascade
# Wrong at Tier 1 → correct after cascade
rescued = cascade_test[(~cascade_test['tier1_correct']) & (cascade_test['final_correct'])]
print(f"\n2. Patients rescued by cascade: {len(rescued)}")
if len(rescued) > 0:
    _d = {CLS_DISP.get(k, k): v for k, v in rescued['true_label'].value_counts().to_dict().items()}
    print(f"   True classes: {_d}")
    print(f"   Tier1 zone distribution: {rescued['tier1_zone'].value_counts().to_dict()}")

# 3. Patients lost after cascade (correct at Tier 1 → wrong after cascade)
lost_by_cascade = cascade_test[(cascade_test['tier1_correct']) & (~cascade_test['final_correct'])]
print(f"\n3. Patients lost by cascade: {len(lost_by_cascade)}")
if len(lost_by_cascade) > 0:
    _d = {CLS_DISP.get(k, k): v for k, v in lost_by_cascade['true_label'].value_counts().to_dict().items()}
    print(f"   True classes: {_d}")

# 4. Per-class loss rate
print(f"\n4. Per-class analysis (Tier 1 → Cascade):")
print(f"   {'Class':<10s} {'n':>4s} {'T1_acc':>7s} {'Cascade_acc':>11s} {'Δ':>7s} {'Rescued':>8s}")
print(f"   {'─'*52}")

classes = ['DAS', 'DEA', 'HA', 'HGB HTZ', 'Normal']
for cls in classes:
    mask = (cascade_test['true_label'] == cls)
    n = mask.sum()
    if n == 0:
        continue
    t1_acc = cascade_test.loc[mask, 'tier1_correct'].mean()
    cas_acc = cascade_test.loc[mask, 'final_correct'].mean()
    n_rescued = ((~cascade_test.loc[mask, 'tier1_correct']) &
                 (cascade_test.loc[mask, 'final_correct'])).sum()
    print(f"   {CLS_DISP[cls]:<10s} {n:>4d} {t1_acc:>7.3f} {cas_acc:>11.3f} {cas_acc-t1_acc:>+7.3f} {n_rescued:>8d}")

# ── Lost patient figure ──
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Left: class distribution of rescued patients
if len(rescued) > 0:
    s2_classes = ['DAS', 'DEA', 'HA', 'HGB HTZ', 'Normal']
    s2_display = ['OAC', 'IDA', 'HA', 'HGB HTZ', 'Normal']
    colors_cls = [PALETTE['base2'], PALETTE['highlight'], PALETTE['base1'],
                  PALETTE['accent1'], PALETTE['accent2']]
    rescue_counts = rescued['true_label'].value_counts().reindex(s2_classes, fill_value=0)
    bars = axes[0].bar(s2_display, rescue_counts.values, color=colors_cls,
                       edgecolor='white', width=0.6)
    for bar, val in zip(bars, rescue_counts.values):
        if val > 0:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[0].set_title(f'Rescued by Cascade ({len(rescued)} patients)',
                      fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Patient count', fontsize=9)
    despine_all(axes[0])

# Right: per-class accuracy Tier1 vs Cascade
x = np.arange(len(classes))
class_display = [CLS_DISP[c] for c in classes]
t1_accs_cls = []
cas_accs_cls = []
for cls in classes:
    mask = (cascade_test['true_label'] == cls)
    t1_accs_cls.append(cascade_test.loc[mask, 'tier1_correct'].mean() if mask.sum() > 0 else 0)
    cas_accs_cls.append(cascade_test.loc[mask, 'final_correct'].mean() if mask.sum() > 0 else 0)

width = 0.35
axes[1].bar(x - width/2, t1_accs_cls, width, color=PALETTE['base1'],
            edgecolor='white', label='Tier 1')
axes[1].bar(x + width/2, cas_accs_cls, width, color=PALETTE['accent3'],
            edgecolor='white', label='Cascade')
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_display, fontsize=8)
axes[1].set_ylabel('Accuracy', fontsize=9)
axes[1].set_title('Per-Class: Tier 1 vs Cascade', fontsize=10, fontweight='bold')
axes[1].legend(fontsize=8, frameon=False)
despine_all(axes[1])

fig.suptitle('Lost Patient Analysis — Cascade Effect (Test)', fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, PATHS.module_dir('m4_e2e', 'plots'), 'cascade_rescue_analysis')
plt.show()
print("\n✅ Figure saved")

## CELL C4 — Cost-Effectiveness Table & Excel Export

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL C4 — Cost-Effectiveness Table & Excel Export
# ══════════════════════════════════════════════════════════

t1 = e2e_results['CBC_Only_test']
t2 = e2e_results['CBC_BIO_test']
n = len(t1)
full_bio_acc = t2['e2e_correct'].mean()
pure_cbc_acc = t1['e2e_correct'].mean()
max_gain = full_bio_acc - pure_cbc_acc

strategies = [
    ('Pure CBC_Only', [], pure_cbc_acc),
    ('LOW only', ['LOW'], None),
    ('MEDIUM+LOW', ['MEDIUM', 'LOW'], None),
    ('MEDIUM+LOW+Excluded', ['MEDIUM', 'LOW', 'Excluded'], None),
    ('Full CBC_BIO', None, full_bio_acc),
]

cost_rows = []
for label, zones, fixed_acc in strategies:
    if fixed_acc is not None and zones is None:
        # Full CBC_BIO
        cost_rows.append({
            'strategy': label, 'acc': round(fixed_acc, 4),
            'bio_pct': 100.0, 'n_bio': n,
            'gain_pct': 100.0, 'delta_acc': round(fixed_acc - pure_cbc_acc, 4)
        })
    elif fixed_acc is not None:
        # Pure CBC_Only
        cost_rows.append({
            'strategy': label, 'acc': round(fixed_acc, 4),
            'bio_pct': 0.0, 'n_bio': 0,
            'gain_pct': 0.0, 'delta_acc': 0.0
        })
    else:
        cascade = simulate_cascade(t1, t2, escalation_zones=zones)
        acc = cascade['final_correct'].mean()
        n_bio = cascade['escalated'].sum()
        gain = (acc - pure_cbc_acc) / max_gain * 100 if max_gain > 0 else 0
        cost_rows.append({
            'strategy': label, 'acc': round(acc, 4),
            'bio_pct': round(n_bio / n * 100, 1), 'n_bio': n_bio,
            'gain_pct': round(gain, 1), 'delta_acc': round(acc - pure_cbc_acc, 4)
        })

cost_df = pd.DataFrame(cost_rows)

print("=" * 75)
print("Cost-Effectiveness Table (Test, n=229)")
print("=" * 75)
print(f"\n{'Strategy':<25s} {'Acc':>6s} {'ΔAcc':>7s} {'BIO %':>7s} {'n_BIO':>6s} {'Gain':>8s}")
print("─" * 65)
for _, r in cost_df.iterrows():
    print(f"  {r['strategy']:<23s} {r['acc']:>6.3f} {r['delta_acc']:>+7.3f} "
          f"{r['bio_pct']:>6.1f}% {r['n_bio']:>5d}   {r['gain_pct']:>6.1f}%")

# ── Cascade metrics Excel ──
with pd.ExcelWriter(reflex_dir / 'cascade_metrics.xlsx') as w:
    cost_df.to_excel(w, sheet_name='cost_effectiveness', index=False)

    # Cascade summary
    for sp in ['oof', 'test']:
        cas = pd.read_parquet(reflex_dir / f'cascade_simulation_{sp}.parquet')
        summary_rows = []
        for z in ['Excluded', 'HIGH', 'MEDIUM', 'LOW']:
            zdf = cas[cas['final_zone'] == z]
            if len(zdf) > 0:
                summary_rows.append({
                    'zone': z, 'n': len(zdf), 'pct': round(len(zdf)/len(cas)*100, 1),
                    'accuracy': round(zdf['final_correct'].mean(), 4)
                })
        pd.DataFrame(summary_rows).to_excel(w, sheet_name=f'cascade_zones_{sp}', index=False)

print("\n✅ cascade_metrics.xlsx")

## CELL C5 — Update src/m4_e2e.py (cascade functions)